In [ ]:
#!/usr/bin/env python
# coding: utf-8
# Stanford RNA 3D Folding 2 - Winning Solution
# Models: Protenix, DRFold2, TBM, RNApro, Boltz2
# Ensemble strategy: model selection based on sequence length

In [ ]:
# ============================================================
# Dependency Installation (Offline Wheels)
# ============================================================
# pip install 
!pip install /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
!pip install /kaggle/input/datasets/yutaroito/boltz-env-depend/boltz_wheels_cuda_v2/gemmi-0.6.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install /kaggle/input/datasets/yutaroito/rdkit-312/rdkit-2025.9.4-cp312-cp312-manylinux_2_28_x86_64.whl
!pip install /kaggle/input/datasets/yutaroito/boltz-env-depend/boltz_wheels_cuda_v2/chembl_structure_pipeline-1.2.2-py3-none-any.whl 
!pip install /kaggle/input/datasets/yutaroito/boltz-env-depend/boltz_wheels_cuda_v2/einx-0.3.0-py3-none-any.whl    

# ============================================================
# Standard library imports
# ============================================================
import builtins
import gc
import json
import logging
import os
import pickle
import random
import shutil
import subprocess
import sys
import tempfile
import time
import traceback
import typing
import warnings
from contextlib import nullcontext
from datetime import datetime
from functools import partial
from pathlib import Path
from timeit import default_timer as timer

warnings.filterwarnings('ignore')
# ============================================================
# Third-party imports
# ============================================================
import numpy as np
import pandas as pd
import pytz
import torch
import torch.nn.functional as F
import yaml
from Bio import pairwise2
from Bio.Align import PairwiseAligner
from Bio.PDB import PDBParser
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.Seq import Seq
from scipy.spatial import distance_matrix
from scipy.spatial.transform import Rotation as R
from tqdm import tqdm
import gemmi
import pytorch_lightning as pl

In [ ]:
# ============================================================
# SECTION 0: Early Exit Guard (Kaggle submission mode check)
# ============================================================

ONLY_INFER = False  # True # False

if ONLY_INFER:
    # --- Early-exit guard for local runs (put this as the very first cell) ---
    flag = os.environ.get("KAGGLE_IS_COMPETITION_RERUN")

    # Kaggle sets this to a truthy value during the submission rerun environment.
    # If not in that environment, write an all-zeros submission and stop.
    if not flag:
        candidate_seq_paths = [
            "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-2/validation_sequences.csv",
            "/kaggle/input/stanford-rna-3d-folding-2/train_sequences.csv",
        ]

        seq_df = None
        for p in candidate_seq_paths:
            if os.path.exists(p):
                seq_df = pd.read_csv(p)
                break

        if seq_df is None or "target_id" not in seq_df.columns or "sequence" not in seq_df.columns:
            raise RuntimeError(
                "Not in competition rerun, and could not find a sequences file to build a valid zero submission.\n"
                "Make sure test_sequences.csv is available under /kaggle/input/... or adjust candidate_seq_paths."
            )

        rows = []
        for row in seq_df.itertuples(index=False):
            tid = row.target_id
            seq = row.sequence
            for i, base in enumerate(seq, start=1):
                rec = {
                    "ID": f"{tid}_{i}",
                    "resname": base,
                    "resid": i,
                }
                for k in range(1, 6):
                    rec[f"x_{k}"] = 0.0
                    rec[f"y_{k}"] = 0.0
                    rec[f"z_{k}"] = 0.0
                rows.append(rec)

        out = pd.DataFrame(rows)

        # Enforce official column order
        cols = (
            ["ID", "resname", "resid"] +
            [f"{ax}_{k}" for k in range(1, 6) for ax in ("x", "y", "z")]
        )
        out = out[cols]
        out.to_csv("submission.csv", index=False)
        print("Not in Kaggle competition rerun. Wrote zero submission.csv and exiting early.")
        raise SystemExit(0)

    # If flag is truthy: do nothing, notebook continues normally.
    print("KAGGLE_IS_COMPETITION_RERUN detected -> continuing normally.")


IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(IS_SCORING_RUN)


subprocess.run(
    ["pip", "install", "--no-deps",
     "/kaggle/input/datasets/tobimichigan/biotite-1-2/biotite-1.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"],
    check=False
)

os.environ["LAYERNORM_TYPE"] = "torch"
os.environ.setdefault("RNA_MSA_DEPTH_LIMIT", "512")

In [ ]:
# ============================================================
# SECTION 1: Protenix Inference (seq_len < 250 or >= 1000)
# ============================================================

# ---- user-configurable paths ----
DEFAULT_TEST_CSV = "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"
DEFAULT_OUTPUT = "/kaggle/working/protenix_submission.csv"
DEFAULT_CODE_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
DEFAULT_ROOT_DIR = "/kaggle/input/datasets/qiweiyin/protenix-v1-adjusted/Protenix-v1-adjust-v2/Protenix-v1-adjust-v2/Protenix-v1"
MODEL_NAME = "protenix_base_20250630_v1.0.0"
input_path = "/kaggle/input/stanford-rna-3d-folding-2"
N_SAMPLE = 5
SEED = 42

MAX_SEQ_LEN = int(os.environ.get("MAX_SEQ_LEN", "512"))
CHUNK_OVERLAP = int(os.environ.get("CHUNK_OVERLAP", "128"))  # 64
MODEL_N_SAMPLE = int(os.environ.get("MODEL_N_SAMPLE", str(N_SAMPLE)))

TEST_TARGETS = False
if not IS_SCORING_RUN:
    TEST_TARGETS = os.environ.get("TEST_TARGETS", "9ZCC").strip()

def parse_bool(value: str, default: bool = False) -> str:
    normalized = str(value).strip().lower()
    if normalized in {"1", "true", "t", "yes", "y", "on"}:
        return "true"
    if normalized in {"0", "false", "f", "no", "n", "off"}:
        return "false"
    return "true" if default else "false"

os.environ["USE_RNA_MSA"] = "true"

USE_MSA = parse_bool(os.environ.get("USE_MSA", "false"))
USE_TEMPLATE = parse_bool(os.environ.get("USE_TEMPLATE", "false"))
USE_RNA_MSA = parse_bool(os.environ.get("USE_RNA_MSA", "false"))
print(f"USE_MSA: {USE_MSA}")
print(f"USE_TEMPLATE: {USE_TEMPLATE}")
print(f"USE_RNA_MSA: {USE_RNA_MSA}")

TRIANGLE_ATTENTION = os.environ.get("TRIANGLE_ATTENTION", "torch").strip().lower()
TRIANGLE_MULTIPLICATIVE = os.environ.get("TRIANGLE_MULTIPLICATIVE", "torch").strip().lower()

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.enabled = True
    torch.use_deterministic_algorithms(True)


def resolve_paths() -> tuple[str, str, str]:
    test_csv = os.environ.get("TEST_CSV", DEFAULT_TEST_CSV)
    output_csv = os.environ.get("SUBMISSION_CSV", DEFAULT_OUTPUT)
    code_dir = os.environ.get("PROTENIX_CODE_DIR", DEFAULT_CODE_DIR)
    root_dir = os.environ.get("PROTENIX_ROOT_DIR", DEFAULT_ROOT_DIR)
    return test_csv, output_csv, code_dir, root_dir


def ensure_required_files(root_dir: str) -> None:
    checkpoint_path = Path(root_dir) / "checkpoint" / f"{MODEL_NAME}.pt"
    components_path = Path(root_dir) / "common" / "components.cif"
    components_pkl_path = Path(root_dir) / "common" / "components.cif.rdkit_mol.pkl"
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {checkpoint_path}.\n"
            "Please include checkpoint/protenix_base_default_v1.0.0.pt in PROTENIX_ROOT_DIR."
        )
    if not components_path.exists():
        raise FileNotFoundError(
            f"Missing CCD file: {components_path}.\n"
            "Please include common/components.cif in PROTENIX_ROOT_DIR."
        )
    if not components_pkl_path.exists():
        raise FileNotFoundError(
            f"Missing CCD cache: {components_pkl_path}.\n"
            "Please include common/components.cif.rdkit_mol.pkl in PROTENIX_ROOT_DIR."
        )


def _detect_seq_type(seq: str) -> str:
    """Return 'rnaSequence', 'dnaSequence', or 'proteinSequence' for Protenix input."""
    non_nucleotide = set(seq.upper()) - set("ACGUTRYSWKMBDHVN")
    if non_nucleotide:
        return "proteinSequence"
    if "T" in seq.upper() and "U" not in seq.upper():
        return "dnaSequence"
    return "rnaSequence"


def _parse_all_sequences(row) -> list[dict]:
    """Parse all_sequences FASTA field into Protenix sequence entries.
    Returns list of {"<seqType>": {"sequence": ..., "count": N}} dicts.
    Each unique chain sequence becomes one entry; copies → count field.
    Falls back to the target RNA sequence alone if parsing fails.
    """
    target_seq = str(row["sequence"])
    try:
        all_seq_raw = str(row.get("all_sequences", "") or "")
        stoich_raw  = str(row.get("stoichiometry", "") or "")
        if not all_seq_raw.strip():
            raise ValueError("no all_sequences")

        # Parse FASTA: {chain_id: sequence_str}
        chains, cur_id, parts = {}, None, []
        for line in all_seq_raw.splitlines():
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    chains[cur_id] = "".join(parts)
                cur_id = line[1:].split()[0]
                parts = []
            else:
                parts.append(line)
        if cur_id is not None:
            chains[cur_id] = "".join(parts)

        # Parse stoichiometry: "A:2;B:1" → {chain_id: count}
        stoich = {}
        for part in stoich_raw.split(";"):
            part = part.strip()
            if ":" in part:
                ch, cnt = part.split(":", 1)
                stoich[ch.strip()] = int(cnt.strip())

        entries = []
        for ch_id, seq in chains.items():
            count = stoich.get(ch_id, 1)
            seq_type = _detect_seq_type(seq)
            entries.append({seq_type: {"sequence": seq, "count": count}})
        return entries if entries else [{"rnaSequence": {"sequence": target_seq, "count": 1}}]
    except Exception:
        seq_type = _detect_seq_type(target_seq)
        return [{seq_type: {"sequence": target_seq, "count": 1}}]


def build_input_json(test_df: pd.DataFrame, json_path: str) -> None:
    data = []
    for _, row in test_df.iterrows():
        target_id = row["target_id"]
        data.append({
            "name": target_id,
            "covalent_bonds": [],
            "sequences": _parse_all_sequences(row),
        })
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_single_input_json(target_id: str, seq: str, json_path: str,
                             row: "pd.Series | None" = None) -> None:
    if row is not None:
        sequences = _parse_all_sequences(row)
    else:
        seq_type = _detect_seq_type(seq)
        sequences = [{seq_type: {"sequence": seq, "count": 1}}]
    data = [{"name": target_id, "covalent_bonds": [], "sequences": sequences}]
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)


def build_configs(input_json_path: str, dump_dir: str, model_name: str):
    from configs.configs_base import configs as configs_base
    from configs.configs_data import data_configs
    from configs.configs_inference import inference_configs
    from configs.configs_model_type import model_configs
    from protenix.config.config import parse_configs

    base_configs = {**configs_base, **{"data": data_configs}, **inference_configs}

    def deep_update(target, patch):
        for key, value in patch.items():
            if isinstance(value, dict) and key in target and isinstance(target[key], dict):
                deep_update(target[key], value)
            else:
                target[key] = value

    deep_update(base_configs, model_configs[model_name])

    arg_str = " ".join(
        [
            f"--model_name {model_name}",
            f"--input_json_path {input_json_path}",
            f"--dump_dir {dump_dir}",
            f"--use_msa {USE_MSA}",
            f"--use_template {USE_TEMPLATE}",
            f"--use_rna_msa {USE_RNA_MSA}",
            f"--sample_diffusion.N_sample {MODEL_N_SAMPLE}",
            f"--seeds {SEED}",
        ]
    )
    configs = parse_configs(
        configs=base_configs,
        arg_str=arg_str,
        fill_required_with_null=True,
    )
    return configs


def get_c1_mask(data: dict, atom_array) -> torch.Tensor:
    if atom_array is not None:
        try:
            if hasattr(atom_array, "centre_atom_mask"):
                centre_mask = atom_array.centre_atom_mask == 1
                if hasattr(atom_array, "is_rna"):
                    centre_mask = centre_mask & (atom_array.is_rna)
                return torch.from_numpy(centre_mask)
            if hasattr(atom_array, "atom_name"):
                if hasattr(atom_array, "is_rna"):
                    return torch.from_numpy(
                        (atom_array.atom_name == "C1'") & (atom_array.is_rna)
                    )
                return torch.from_numpy(atom_array.atom_name == "C1'")
        except Exception:
            pass

    features = data["input_feature_dict"]
    if "centre_atom_mask" in features:
        return features["centre_atom_mask"].long() == 1
    return features["atom_to_tokatom_idx"].long() == 12


def get_feature_c1_mask(data: dict) -> torch.Tensor:
    features = data["input_feature_dict"]
    if "centre_atom_mask" in features:
        return features["centre_atom_mask"].long() == 1
    return features["atom_to_tokatom_idx"].long() == 12


def coords_to_rows(target_id: str, seq: str, coords: np.ndarray) -> list[dict]:
    rows = []
    n_res = len(seq)
    for i in range(n_res):
        row = {
            "ID": f"{target_id}_{i + 1}",
            "resname": seq[i],
            "resid": i + 1,
        }
        for s in range(N_SAMPLE):
            if s < coords.shape[0] and i < coords.shape[1]:
                x, y, z = coords[s, i]
            else:
                x, y, z = 0.0, 0.0, 0.0
            row[f"x_{s + 1}"] = float(x)
            row[f"y_{s + 1}"] = float(y)
            row[f"z_{s + 1}"] = float(z)
        rows.append(row)
    return rows


def pad_samples(coords: np.ndarray, target_samples: int) -> np.ndarray:
    if coords.shape[0] >= target_samples:
        return coords
    if coords.shape[0] == 0:
        return np.zeros((target_samples, coords.shape[1], 3), dtype=coords.dtype)
    repeat = target_samples - coords.shape[0]
    extra = np.repeat(coords[:1], repeat, axis=0)
    return np.concatenate([coords, extra], axis=0)


def split_sequence(seq: str, max_len: int, overlap: int) -> list[tuple[int, int, str]]:
    if max_len <= 0:
        raise ValueError("max_len must be > 0")
    if overlap >= max_len:
        raise ValueError("overlap must be smaller than max_len")
    if len(seq) <= max_len:
        return [(0, len(seq), seq)]
    step = max_len - overlap
    chunks = []
    start = 0
    while start < len(seq):
        end = min(start + max_len, len(seq))
        chunks.append((start, end, seq[start:end]))
        if end == len(seq):
            break
        start += step
    return chunks

def main_protenix() -> None:
    test_csv, output_csv, code_dir, root_dir = resolve_paths()

    if not os.path.isdir(code_dir):
        raise FileNotFoundError(
            f"Missing PROTENIX_CODE_DIR: {code_dir}. Set PROTENIX_CODE_DIR to the repo path."
        )

    os.environ["PROTENIX_ROOT_DIR"] = root_dir
    sys.path.append(code_dir)

    ensure_required_files(root_dir)
    seed_everything(SEED)

    test_df = pd.read_csv(test_csv)
    test_df["sequence_len"] = test_df["sequence"].str.len()
    test_df = test_df[(test_df['sequence_len'] < 250) | (test_df["sequence_len"] >= 1000)]

    if not IS_SCORING_RUN:
        if TEST_TARGETS:
            keep = {t.strip() for t in TEST_TARGETS.split(",") if t.strip()}
            test_df = test_df[test_df["target_id"].isin(keep)].reset_index(drop=True)
            if test_df.empty:
                raise ValueError("TEST_TARGETS did not match any target_id values")

    work_dir = Path("/kaggle/working")

    work_dir.mkdir(parents=True, exist_ok=True)
    input_json_path = str(work_dir / "protenix_input.json")
    build_input_json(test_df, input_json_path)

    chunks_dir = work_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)

    from protenix.data.inference.infer_dataloader import InferenceDataset
    from runner.inference import InferenceRunner, update_gpu_compatible_configs, update_inference_configs

    configs = build_configs(input_json_path, str(work_dir / "outputs"), MODEL_NAME)
    configs = update_gpu_compatible_configs(configs)
    runner = InferenceRunner(configs)

    seq_by_id = dict(zip(test_df.target_id.tolist(), test_df.sequence.tolist()))

    all_predictions = []

    debug_path = Path(output_csv).with_suffix(".debug.csv")
    debug_rows = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
        target_id = row["target_id"]
        seq = row["sequence"]

        if len(seq) > MAX_SEQ_LEN:
            chunks = split_sequence(seq, MAX_SEQ_LEN, CHUNK_OVERLAP)
            print(
                f"[{target_id}] long sequence: {len(seq)} -> {len(chunks)} chunks "
                f"(max_len={MAX_SEQ_LEN}, overlap={CHUNK_OVERLAP})"
            )
            full_coords = np.zeros((N_SAMPLE, len(seq), 3), dtype=np.float32)
            for chunk_idx, (start, end, subseq) in enumerate(chunks):
                chunk_id = f"{target_id}__chunk{chunk_idx:03d}"
                chunk_json_path = str(chunks_dir / f"{chunk_id}.json")
                build_single_input_json(chunk_id, subseq, chunk_json_path)
                chunk_configs = build_configs(
                    chunk_json_path, str(work_dir / "outputs"), MODEL_NAME
                )
                chunk_configs = update_gpu_compatible_configs(chunk_configs)
                chunk_dataset = InferenceDataset(chunk_configs)
                chunk_data, chunk_atom_array, chunk_error = chunk_dataset[0]

                if chunk_error:
                    print(f"[{chunk_id}] data_error: {chunk_error.splitlines()[0]}")
                    debug_rows.append(
                        {
                            "target_id": target_id,
                            "seq_len": len(seq),
                            "c1_count": 0,
                            "is_rna_atoms": -1,
                            "coord_abs_max": 0.0,
                            "error": chunk_error,
                            "split_mode": "chunked",
                            "chunk_idx": chunk_idx,
                            "chunk_start": start,
                            "chunk_end": end,
                            "chunk_len": end - start,
                        }
                    )
                    continue

                new_configs = update_inference_configs(
                    configs, chunk_data["N_token"].item()
                )
                runner.update_model_configs(new_configs)

                prediction = runner.predict(chunk_data)
                coords = prediction["coordinate"]
                mask = get_c1_mask(chunk_data, chunk_atom_array)
                c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                if c1_count == 0:
                    mask = get_feature_c1_mask(chunk_data)
                    c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                    print(
                        f"[{chunk_id}] C1' atoms: {c1_count} (seq_len={len(subseq)}) fallback"
                    )
                else:
                    print(
                        f"[{chunk_id}] C1' atoms: {c1_count} (seq_len={len(subseq)})"
                    )
                is_rna_count = (
                    int(chunk_atom_array.is_rna.sum())
                    if hasattr(chunk_atom_array, "is_rna")
                    else -1
                )
                coord_abs_max = float(coords.abs().max().item())
                debug_rows.append(
                    {
                        "target_id": target_id,
                        "seq_len": len(seq),
                        "c1_count": c1_count,
                        "is_rna_atoms": is_rna_count,
                        "coord_abs_max": coord_abs_max,
                        "error": "",
                        "split_mode": "chunked",
                        "chunk_idx": chunk_idx,
                        "chunk_start": start,
                        "chunk_end": end,
                        "chunk_len": end - start,
                    }
                )
                coords_all = prediction["coordinate"]  # (S, N_atom, 3)

                res_ids = chunk_atom_array.res_id
                atom_names = chunk_atom_array.atom_name

                c1_indices = []
                for r in range(1, len(subseq) + 1):
                    idx = np.where(
                        (res_ids == r) &
                        (atom_names == "C1'")
                    )[0]
                    if len(idx) == 0:
                        continue
                    c1_indices.append(idx[0])

                coords = coords_all[:, c1_indices, :]
                coords = coords.detach().cpu().numpy()

                if coords.shape[1] != len(subseq):
                    padded = np.zeros((coords.shape[0], len(subseq), 3), dtype=np.float32)
                    min_len = min(coords.shape[1], len(subseq))
                    if min_len > 0:
                        padded[:, :min_len, :] = coords[:, :min_len, :]
                    coords = padded

                coords = pad_samples(coords, N_SAMPLE)

                full_coords[:, start:end, :] = coords[:, : end - start, :]
                del prediction, coords, mask, chunk_data, chunk_atom_array, chunk_dataset
                torch.cuda.empty_cache()
                gc.collect()

            rows = coords_to_rows(target_id, seq, full_coords)
            all_predictions.extend(rows)
            continue

        elif len(seq) <= MAX_SEQ_LEN:
            single_json = str(work_dir / f"{target_id}.json")
            build_single_input_json(target_id, seq, single_json)

            single_configs = build_configs(
                single_json,
                str(work_dir / "outputs"),
                MODEL_NAME,
            )
            single_configs = update_gpu_compatible_configs(single_configs)

            dataset = InferenceDataset(single_configs)
            data, atom_array, error_message = dataset[0]
            if error_message:
                print(f"[{target_id}] data_error: {error_message.splitlines()[0]}")
                debug_rows.append(
                    {
                        "target_id": target_id,
                        "seq_len": len(seq),
                        "c1_count": 0,
                        "is_rna_atoms": -1,
                        "coord_abs_max": 0.0,
                        "error": error_message,
                        "split_mode": "full",
                        "chunk_idx": -1,
                        "chunk_start": 0,
                        "chunk_end": len(seq),
                        "chunk_len": len(seq),
                    }
                )
                rows = coords_to_rows(
                    target_id,
                    seq,
                    np.zeros((0, 0, 3), dtype=np.float32),
                )
                all_predictions.extend(rows)
                continue

            new_configs = update_inference_configs(configs, data["N_token"].item())
            runner.update_model_configs(new_configs)

            prediction = runner.predict(data)
            coords = prediction["coordinate"]
            mask = get_c1_mask(data, atom_array)
            c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
            if c1_count == 0:
                mask = get_feature_c1_mask(data)
                c1_count = int(mask.sum().item()) if hasattr(mask, "sum") else 0
                print(
                    f"[{target_id}] C1' atoms: {c1_count} (seq_len={len(seq)}) fallback"
                )
            else:
                print(f"[{target_id}] C1' atoms: {c1_count} (seq_len={len(seq)})")
            is_rna_count = (
                int(atom_array.is_rna.sum()) if hasattr(atom_array, "is_rna") else -1
            )
            coord_abs_max = float(coords.abs().max().item())
            debug_rows.append(
                {
                    "target_id": target_id,
                    "seq_len": len(seq),
                    "c1_count": c1_count,
                    "is_rna_atoms": is_rna_count,
                    "coord_abs_max": coord_abs_max,
                    "error": "",
                    "split_mode": "full",
                    "chunk_idx": -1,
                    "chunk_start": 0,
                    "chunk_end": len(seq),
                    "chunk_len": len(seq),
                }
            )

            coords_all = prediction["coordinate"]  # (S, N_atom, 3)

            res_ids = atom_array.res_id
            atom_names = atom_array.atom_name

            c1_indices = []
            for r in range(1, len(seq) + 1):
                idx = np.where(
                    (res_ids == r) &
                    (atom_names == "C1'")
                )[0]
                if len(idx) == 0:
                    continue
                c1_indices.append(idx[0])

            coords = coords_all[:, c1_indices, :]
            coords = coords.detach().cpu().numpy()

            if coords.shape[1] != len(seq):
                padded = np.zeros((coords.shape[0], len(seq), 3), dtype=np.float32)
                min_len = min(coords.shape[1], len(seq))
                if min_len > 0:
                    padded[:, :min_len, :] = coords[:, :min_len, :]
                coords = padded

            coords = pad_samples(coords, N_SAMPLE)

            rows = coords_to_rows(target_id, seq, coords)
            all_predictions.extend(rows)
            del prediction, coords, mask, data, atom_array
            torch.cuda.empty_cache()
            gc.collect()

    if debug_rows:
        pd.DataFrame(debug_rows).to_csv(debug_path, index=False)

    sub = pd.DataFrame(all_predictions)
    cols = ["ID", "resname", "resid"] + [
        f"{c}_{i}" for i in range(1, N_SAMPLE + 1) for c in ["x", "y", "z"]
    ]
    coord_cols = [c for c in cols if c.startswith(("x_", "y_", "z_"))]
    sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)
    sub[cols].to_csv(output_csv, index=False)

    print(f"Saved submission to {output_csv}")


if __name__ == "__main__":
    main_protenix()


protenix_pred = pd.read_csv('/kaggle/working/protenix_submission.csv')

In [ ]:
# ============================================================
# SECTION 2: DRFold2 Inference (seq_len < 250)
# ============================================================

test_sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")

is_submission_mode = len(test_sequences) != 12


os.environ['CUDA_VISIBLE_DEVICES'] = '0'

sys.argv = ['notebook', 'cuda' if torch.cuda.is_available() else 'cpu', 'fp32'] #'fp16' #'fp32'# 'bf16' #
device = sys.argv[1]
sys_dtype = sys.argv[2] if len(sys.argv) > 2 else 'fp32'

print('Using device:', device)
print('Using dtype:', sys_dtype)

# dr settings
NUM_CONF=5
MAX_LENGTH=480
MAX_CAT_LENGTH=2400

CFG_DIR='cfg_97'

DEVICE=device #'cuda' #'cpu'#
PREC=sys_dtype 


if PREC=='fp16':
    torch.set_default_dtype(torch.float16)
if PREC=='bf16':
    torch.set_default_dtype(torch.bfloat16)


from datetime import datetime
import pytz
print('LOGGING TIME OF START:',  datetime.strftime(datetime.now(pytz.timezone('Asia/Singapore')), "%Y-%m-%d %H:%M:%S"))


print('PIP INSTALL OK !!!!')
import os,sys

import pandas as pd
pd.set_option('display.max_columns', 20)
pd.set_option('display.expand_frame_repr', False)

import numpy as np
import torch
import torch.nn.functional as F
from timeit import default_timer as timer


# helper--
class dotdict(dict):
	__setattr__ = dict.__setitem__
	__delattr__ = dict.__delitem__

	def __getattr__(self, name):
		try:
			return self[name]
		except KeyError:
			raise AttributeError(name)

def time_to_str(t, mode='min'):
	if mode=='min':
		t  = int(t)/60
		hr = t//60
		min = t%60
		return '%2d hr %02d min'%(hr,min) 
	elif mode=='sec':
		t   = int(t)
		min = t//60
		sec = t%60
		return '%2d min %02d sec'%(min,sec)

	else:
		raise NotImplementedError

def gpu_memory_use():
    if torch.cuda.is_available():
        device = torch.device(0)
        free, total = torch.cuda.mem_get_info(device)
        used= (total - free) / 1024 ** 3
        return round(used,2)
    else:
        return 0

print('torch',torch.__version__)
print('torch.cuda',torch.version.cuda)

print('IMPORT OK!!!')

DATA_KAGGLE_DIR = '/kaggle/input/stanford-rna-3d-folding-2'

valid_df = pd.read_csv(f'{DATA_KAGGLE_DIR}/test_sequences.csv')
# Sort test sequences by length to process shorter ones with DRfold2
valid_df["sequence_len"] = valid_df["sequence"].str.len()
valid_df = valid_df[valid_df['sequence_len'] < 250]
valid_df.reset_index(drop=True, inplace=True)
print(valid_df.shape)
 
if not IS_SCORING_RUN:
    valid_df = valid_df.head(5)

print('len(valid_df)',len(valid_df))
print(valid_df.iloc[0])
print('')


print('SETTING OK!!!')


sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold')
import a2b
def frame_coor_to_C1(coor, seq, BASE_COOR, OTHER_COOR):
    tx = torch.as_tensor(coor, dtype=torch.float32)
    basex = torch.from_numpy(Get_base(seq, BASE_COOR)).to(tx.dtype)
    otherx = torch.from_numpy(Get_base(seq, OTHER_COOR)).to(tx.dtype)
    L = len(seq)

    x = torch.rand((L, 21), dtype=tx.dtype, device=tx.device)

    biasq = tx.mean(dim=1, keepdim=True)            # (L, 1, 3)
    q = tx - biasq                                  # (L, 3, 3)

    m = torch.einsum('bnz,bny->bzy', basex, q).reshape(L, -1)
    x[:, :9] = m
    x[:, 9:18] = m

    x[:, 18:] = biasq.squeeze(1)
    rama = x.double()  

    #    otherx: (L, 5, 3) -> other_xyz: (L, 5, 3)
    other_xyz = a2b.quat2b(otherx.double(), rama[:, 9:]).float().cpu().numpy()

    c1_xyz = other_xyz[:, 4, :]
    return c1_xyz


def concat_coor(out1: dict, out2: dict) -> np.ndarray:
    coor1 = torch.as_tensor(out1['coor'], dtype=torch.float64)   # (L1,3,3)
    coor2 = torch.as_tensor(out2['coor'], dtype=torch.float64)   # (L2,3,3)

    f1 = coor1[-1]   # (3,3)
    f2 = coor2[0]    # (3,3)

    bias1 = f1.mean(dim=0)   # (3,)
    bias2 = f2.mean(dim=0)   # (3,)
    basex = f1 - bias1       # (3,3)
    q     = f2 - bias2       # (3,3)

    #    R_{ij} = sum_z basex_{iz} * q_{jz}
    R = torch.einsum('iz,jz->ij', basex, q)   # (3,3)

    t = bias1 - (R @ bias2)                  # (3,)

    L2 = coor2.shape[0]
    rama = torch.empty((L2, 12), dtype=torch.float64, device=coor2.device)
    R_flat = R.reshape(1, 9).repeat(L2, 1)    # (L2,9)
    t_rep  = t.reshape(1, 3).repeat(L2, 1)    # (L2,3)
    rama[:, :9] = R_flat
    rama[:, 9:] = t_rep

    coor2_aligned = a2b.quat2b(coor2, rama)   # torch.Tensor (L2,3,3)

    coor_cat = torch.cat([coor1, coor2_aligned[1:]], dim=0)  # (L1+L2-1,3,3)

    return coor_cat.cpu().numpy()


def Get_base(seq, basenpy_standard):
    n_atoms = basenpy_standard.shape[1]
    basenpy = np.zeros([len(seq), n_atoms, 3])
    seqnpy = np.array(list(seq))
    basenpy[seqnpy=='A'] = basenpy_standard[0]
    basenpy[seqnpy=='a'] = basenpy_standard[0]
    basenpy[seqnpy=='G'] = basenpy_standard[1]
    basenpy[seqnpy=='g'] = basenpy_standard[1]
    basenpy[seqnpy=='C'] = basenpy_standard[2]
    basenpy[seqnpy=='c'] = basenpy_standard[2]
    basenpy[seqnpy=='U'] = basenpy_standard[3]
    basenpy[seqnpy=='u'] = basenpy_standard[3]
    basenpy[seqnpy=='T'] = basenpy_standard[3]
    basenpy[seqnpy=='t'] = basenpy_standard[3]
    return basenpy


import numpy as np

def score_energy_one_simple(seq, target_id, out):

    coor = out['coor']  # (L, 3, 3)
    L = len(seq)

    d0_P_C4  = 1.60
    d0_C4_N  = 1.47
    k_bond   = 100.0 

    energy_bond = 0.0
    for i in range(L):
        P  = coor[i,0]
        C4 = coor[i,1]
        N  = coor[i,2]
        d_PC4 = np.linalg.norm(P - C4)
        d_C4N = np.linalg.norm(C4 - N)
        energy_bond += k_bond * (d_PC4 - d0_P_C4)**2
        energy_bond += k_bond * (d_C4N - d0_C4_N)**2

    theta0   = np.deg2rad(109.5)
    k_angle  = 20.0   

    energy_angle = 0.0
    for i in range(L):
        P  = coor[i,0]
        C4 = coor[i,1]
        N  = coor[i,2]
        v1 = P  - C4
        v2 = N  - C4
        cos_theta = np.dot(v1, v2) / (np.linalg.norm(v1)*np.linalg.norm(v2) + 1e-8)
        theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        energy_angle += k_angle * (theta - theta0)**2

    d0_stack = 3.4
    k_stack  = 5.0   # (kcal/mol/Å²)

    energy_stack = 0.0
    for i in range(L-1):
        C4_i   = coor[i  ,1]
        C4_ip1 = coor[i+1,1]
        d = np.linalg.norm(C4_i - C4_ip1)
        energy_stack += k_stack * (d - d0_stack)**2

    total_energy = energy_bond + energy_angle + energy_stack

    # print(f"[{target_id}] bond={energy_bond:.2f}, angle={energy_angle:.2f}, stack={energy_stack:.2f} → total={total_energy:.2f}")

    return total_energy


import numpy as np
import os
import json
import pickle
import numpy as np
import tempfile
from Bio.PDB import PDBParser
sys.path.append('/kaggle/input/drfold-model/DRfold2/PotentialFold')
from Optimization import Structure

sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16')
sys.path.append('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold')
sys.path.append(f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/{CFG_DIR}')
sys.path.append(f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/{CFG_DIR}/RNALM2')

BASE_COOR  = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/base.npy')
OTHER_COOR = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/other2.npy')
SIDE_COOR  = np.load('/kaggle/input/datasets/z1493916656/drfold-model-bf16/PotentialFold/lib/side.npy')

# ---- Full-atom PDB writer (from cfg_97/data.py, uses notebook globals) ----
def write_drfold_pdb(coor, seq, pdbfile):
    """Write full-atom PDB from DRFold2 backbone frames.
    Uses BASE_COOR / OTHER_COOR / SIDE_COOR loaded above.
    Atoms written per residue:
      Frame : P, C4', N9(purine)/N1(pyrimidine)   [3 atoms]
      Other : O5', C5', C3', O3', C1'             [5 atoms]
      Side  : N1,C2,O2,N2,N3,N4,C4,O4,C5,C6,O6,N6,N7,N8,N9  [15 atoms]
    """
    import torch as _torch
    _tx    = torch.from_numpy(coor.astype(np.float64))
    _basex = torch.from_numpy(Get_base(seq, BASE_COOR).astype(np.float64))
    _othx  = torch.from_numpy(Get_base(seq, OTHER_COOR).astype(np.float64))
    _sidx  = torch.from_numpy(Get_base(seq, SIDE_COOR).astype(np.float64))
    L = len(seq)
    _x = torch.zeros((L, 21), dtype=torch.float64)
    _biasq = _tx.mean(dim=1, keepdim=True)
    _q = _tx - _biasq
    _m = torch.einsum('bnz,bny->bzy', _basex, _q).reshape(L, -1)
    _x[:, :9] = _m; _x[:, 9:18] = _m
    _x[:, 18:] = _biasq.squeeze(1)
    _rama = _x
    _xyz      = a2b.quat2b(_basex, _rama[:, 9:]).float().numpy()  # (L,3,3)
    _other    = a2b.quat2b(_othx,  _rama[:, 9:]).float().numpy()  # (L,5,3)
    _side     = a2b.quat2b(_sidx,  _rama[:, 9:]).float().numpy()  # (L,15,3)
    _frame_names  = {'purine':  [' P  ', " C4'", ' N9 '],
                     'pyrimidine': [' P  ', " C4'", ' N1 ']}
    _frame_elem   = ['P', 'C', 'N']
    _other_names  = [" O5'", " C5'", " C3'", " O3'", " C1'"]
    _other_elem   = ['O', 'C', 'C', 'O', 'C']
    _side_names   = [' N1 ',' C2 ',' O2 ',' N2 ',' N3 ',' N4 ',
                     ' C4 ',' O4 ',' C5 ',' C6 ',' O6 ',' N6 ',
                     ' N7 ',' N8 ',' N9 ']
    _side_elem    = ['N','C','O','N','N','N','C','O','C','C','O','N','N','N','N']
    _fmt = '%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
    lines = ['REMARK DRFold2 full-atom reconstruction']
    cnt = 1
    for i in range(L):
        nt = seq[i].upper()
        _fn = _frame_names['purine' if nt in 'AG' else 'pyrimidine']
        for j in range(3):
            lines.append(_fmt % ('ATOM  ', cnt, _fn[j], nt, 'A', i+1,
                _xyz[i,j,0], _xyz[i,j,1], _xyz[i,j,2], 0, 0, _frame_elem[j], ''))
            cnt += 1
        for j in range(5):
            lines.append(_fmt % ('ATOM  ', cnt, _other_names[j], nt, 'A', i+1,
                _other[i,j,0], _other[i,j,1], _other[i,j,2], 0, 0, _other_elem[j], ''))
            cnt += 1
        for j in range(15):
            if _side[i,j].any():  # skip zero-padded atoms
                lines.append(_fmt % ('ATOM  ', cnt, _side_names[j], nt, 'A', i+1,
                    _side[i,j,0], _side[i,j,1], _side[i,j,2], 0, 0, _side_elem[j], ''))
                cnt += 1
    lines.append('END')
    with open(pdbfile, 'w') as _f:
        _f.write('\n'.join(lines))


from EvoMSA2XYZ import MSA2XYZ
from RNALM2.Model import RNA2nd
from data import parse_seq


# data helper
def make_data(seq, device):
    aa_type = parse_seq(seq)
    base = Get_base(seq, BASE_COOR)
    seq_idx = np.arange(len(seq)) + 1

    msa = aa_type[None, :]
    msa = torch.from_numpy(msa)
    msa = torch.cat([msa, msa], 0)  # ???
    msa = F.one_hot(msa.long(), 6).float()

    base_x = torch.from_numpy(base).float()
    seq_idx = torch.from_numpy(seq_idx).long()

    msa, base_x, seq_idx = msa.to(device), base_x.to(device), seq_idx.to(device)
    return msa, base_x, seq_idx


def coord_to_df(sequence, coord, target_id):
    L = len(sequence)
    df = pd.DataFrame()
    df['ID'] = [f'{target_id}_{i + 1}' for i in range(L)]
    df['resname'] = [s for s in sequence]
    df['resid'] = [i + 1 for i in range(L)]

    num_coord = len(coord)
    for j in range(num_coord):
        df[f'x_{j+1}'] = coord[j][:, 0]
        df[f'y_{j+1}'] = coord[j][:, 1]
        df[f'z_{j+1}'] = coord[j][:, 2]
    return df


import warnings
warnings.filterwarnings("ignore")

def run_submit(valid_df):
    
    #load model (these are moified versions, not the same from their github repo)
    rnalm = RNA2nd(dict(
        s_in_dim=5,
        z_in_dim=2,
        s_dim= 512,
        z_dim= 128,
        N_elayers=18,
    ))
    rnalm_file = '/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/RCLM/epoch_67000'
    print(rnalm_file)
    print(
        rnalm.load_state_dict(torch.load(rnalm_file, map_location='cpu', weights_only=True), strict=False)
        #Unexpected key(s) in state_dict: "ss_head.linear.weight", "ss_head.linear.bias".
    )
    rnalm = rnalm.to(DEVICE)
    if(PREC=='fp16'):
        rnalm=rnalm.half()
        
    if PREC=='bf16':
        rnalm = rnalm.bfloat16()
        
    rnalm = rnalm.eval()

    #---
    msa2xyz = MSA2XYZ(
        seq_dim=6,
        msa_dim=7,
        N_ensemble=1,
        N_cycle=8,  # 8
        m_dim=64,
        s_dim=64,
        z_dim=64,
    )
    msa2xyz_file = [f'/kaggle/input/datasets/z1493916656/drfold-model-bf16/model_hub/{CFG_DIR}/model_{i}' for i in range(20)]
    num_msa2xyz = len(msa2xyz_file) 
    msa2xyz_state_dict = []
    for c in range(num_msa2xyz):
        if c==0: print(msa2xyz_file[c])
        m = torch.load(msa2xyz_file[c], map_location='cpu', weights_only=True)
        msa2xyz_state_dict.append(m)
        
    print(msa2xyz.load_state_dict(msa2xyz_state_dict[0], strict=False))
    msa2xyz = msa2xyz.to(DEVICE)
    if(PREC=='fp16'):
        msa2xyz=msa2xyz.half()

    if PREC=='bf16':
        msa2xyz = msa2xyz.bfloat16()
    msa2xyz = msa2xyz.eval()
    
    msa2xyz.msaxyzone.premsa.rnalm = rnalm

    #---


    drfold_pred = [] 
    total_time_taken = 0
    max_gpu_mem_used = 0

    for i, row in valid_df.iterrows():
        start_timer = timer()
        target_id = row.target_id  # 'R1116' #casp15 R1116: len(157)
        sequence = row.sequence
        seq = row.sequence  
        L = len(seq)
        if L > MAX_CAT_LENGTH:
            seq = seq[:MAX_CAT_LENGTH]
        print(i, target_id, L, len(seq), seq[:75] + '...')

        
        if len(seq)>480:
            model_to_try=[16, 9, 1, 2, 0]
        elif len(seq)>200:
            model_to_try = [13, 6, 14, 5, 3]
        elif  len(seq)>100:
            model_to_try = [13, 6, 14, 12, 7, 2, 5, 19, 10, 9]
        else:

            model_to_try = [1, 2, 0, 8, 7, 5, 6, 14, 10, 18, 4, 13, 3, 17, 19, 11, 12, 15, 16, 9]


        # Segment prediction for long sequences
        def predict_segment(seq):
            msa, base_x, seq_idx = make_data(seq, DEVICE)
            with torch.no_grad():
                if PREC=='fp16':
                    msa, base_x = msa.half(), base_x.half()
                if PREC=='bf16':
                    msa, base_x = msa.bfloat16(), base_x.bfloat16()
                return msa2xyz.pred(msa, seq_idx, None, base_x, np.array(list(seq)))

                
        energy = []
        coordinate=[]
        outputs=[]
        for c in model_to_try:
            msa2xyz.load_state_dict(msa2xyz_state_dict[c], strict=False)

            if len(seq) <= MAX_LENGTH:
                outs = [ predict_segment(seq) ]
            else:
                step = MAX_LENGTH - 1
                outs = []
                for s in range(0, len(seq), step):
                    seg = seq[s : min(s+MAX_LENGTH, len(seq))]
                    outs.append(predict_segment(seg))
                    
            out_cat = outs[0]
            for out_seg in outs[1:]:
                out_cat = {'coor': concat_coor(out_cat, out_seg)}
                    
            e = score_energy_one_simple(seq, target_id, out_cat)
            energy.append(e) #tranucated sequence
            
            if L != len(seq):
                out_cat['coor'] = np.pad(out_cat['coor'], ((0, L - len(seq)), (0, 0), (0, 0)), 'constant', constant_values=0)
                
            outputs.append(out_cat)
            
            
            xyz = frame_coor_to_C1(out_cat['coor'], sequence, BASE_COOR, OTHER_COOR)
            
            coordinate.append(xyz)
            

            time_taken = timer() - start_timer
            total_time_taken += time_taken
    
            gpu_mem_used = gpu_memory_use()
            max_gpu_mem_used = max(max_gpu_mem_used,gpu_mem_used)
    
            print(f'{c:02d}   energy:{e:10.0f}   out_cat{str(out_cat["coor"].shape)}  time:{time_to_str(time_taken, mode="sec")}   gpu={gpu_mem_used} gb')

            
        #------- 
        torch.cuda.empty_cache()
        
        energy = np.array(energy)
        energy_mean = np.mean(energy)
        energy = np.abs(energy - energy_mean)
        #select top5
        argsort = np.array(energy).argsort()
        argsort = argsort[:5]

        # Write full-atom PDB for each of the top-5 selected models
        _pdb_dir = Path(f'/kaggle/working/drfold_structures/{target_id}')
        _pdb_dir.mkdir(parents=True, exist_ok=True)
        for _rank, _k in enumerate(argsort, 1):
            _pdb_path = str(_pdb_dir / f'{target_id}_model_{_rank}.pdb')
            try:
                write_drfold_pdb(outputs[_k]['coor'], sequence, _pdb_path)
            except Exception as _e:
                print(f'  PDB write failed for {target_id} rank {_rank}: {_e}')

        df = coord_to_df(row.sequence, [coordinate[k] for k in argsort], row.target_id)
        drfold_pred.append(df)
    
    print('----------------------------------------')
    print('MAX_LENGTH', MAX_LENGTH)
    print('### total_time_taken:', time_to_str(total_time_taken, mode='min'))
    print('### max_gpu_mem_used:', max_gpu_mem_used, 'GB')
    print('')

    drfold_pred = pd.concat(drfold_pred)
    drfold_pred.to_csv(f'/kaggle/working/drfold_submission.csv', index=False)
    print(drfold_pred)
    return drfold_pred

run_submit(valid_df)

print('SUBMIT OK!!!')


drfold_pred = pd.read_csv('/kaggle/working/drfold_submission.csv')

In [ ]:
# ============================================================
# SECTION 3: TBM (Template-Based Modeling, all sequences)
# ============================================================

!pip install --no-index /kaggle/input/datasets/kami1976/biopython-cp312/biopython-1.86-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl


import pandas as pd
import numpy as np
import random
import time
import warnings
import os, sys

warnings.filterwarnings('ignore')

DATA_PATH = '/kaggle/input/stanford-rna-3d-folding-2/'
train_seqs = pd.read_csv(DATA_PATH + 'train_sequences.csv')
test_seqs = pd.read_csv(DATA_PATH + 'test_sequences.csv')
train_labels = pd.read_csv(DATA_PATH + 'train_labels.csv')

sys.path.append(os.path.join(DATA_PATH, "extra"))

# --- Robust import for Kaggle's extra/parse_fasta_py.py (it may miss typing imports) ---
try:
    import typing as _typing
    import builtins as _builtins

    # Make these names available during module import-time annotation evaluation
    _builtins.Dict  = getattr(_typing, "Dict")
    _builtins.Tuple = getattr(_typing, "Tuple")
    _builtins.List  = getattr(_typing, "List")

    from parse_fasta_py import parse_fasta as _parse_fasta_raw

    # Normalize output to: {chain_id: sequence_string}
    def parse_fasta(fasta_content: str):
        d = _parse_fasta_raw(fasta_content)
        out = {}
        for k, v in d.items():
            # some variants return (sequence, headers/lines) or similar
            out[k] = v[0] if isinstance(v, tuple) else v
        return out

except Exception:
    # Fallback FASTA parser: {chain_id: sequence_string}
    def parse_fasta(fasta_content: str):
        out = {}
        cur = None
        seq_parts = []
        for line in str(fasta_content).splitlines():
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur is not None:
                    out[cur] = "".join(seq_parts)
                header = line[1:]
                # First token is usually chain id in this dataset
                cur = header.split()[0]
                seq_parts = []
            else:
                seq_parts.append(line.replace(" ", ""))
        if cur is not None:
            out[cur] = "".join(seq_parts)
        return out

def parse_stoichiometry(stoich: str):
    if pd.isna(stoich) or str(stoich).strip() == "":
        return []
    out = []
    for part in str(stoich).split(';'):
        ch, cnt = part.split(':')
        out.append((ch.strip(), int(cnt)))
    return out

def get_chain_segments(row):
    """
    Returns list of (start,end) segments in row['sequence'] corresponding to chain copies in stoichiometry order.
    Falls back to single segment if parsing fails.
    """
    seq = row['sequence']
    stoich = row.get('stoichiometry', '')
    all_seq = row.get('all_sequences', '')

    if pd.isna(stoich) or pd.isna(all_seq) or str(stoich).strip()=="" or str(all_seq).strip()=="":
        return [(0, len(seq))]

    try:
        chain_dict = parse_fasta(all_seq)  # dict: chain_id -> sequence
        order = parse_stoichiometry(stoich)
        segs = []
        pos = 0
        for ch, cnt in order:
            base = chain_dict.get(ch)
            if base is None:
                return [(0, len(seq))]
            for _ in range(cnt):
                L = len(base)
                segs.append((pos, pos + L))
                pos += L
        if pos != len(seq):
            return [(0, len(seq))]
        return segs
    except Exception:
        return [(0, len(seq))]

def build_segments_map(df):
    seg_map = {}
    stoich_map = {}
    for _, r in df.iterrows():
        tid = r['target_id']
        seg_map[tid] = get_chain_segments(r)
        stoich_map[tid] = str(r.get('stoichiometry', '') if not pd.isna(r.get('stoichiometry', '')) else '')
    return seg_map, stoich_map

train_segs_map, train_stoich_map = build_segments_map(train_seqs)
test_segs_map,  test_stoich_map  = build_segments_map(test_seqs)

def process_labels(labels_df):
    coords_dict = {}
    # Faster + safer prefix extraction
    prefixes = labels_df['ID'].str.rsplit('_', n=1).str[0]
    for id_prefix, group in labels_df.groupby(prefixes):
        coords_dict[id_prefix] = group.sort_values('resid')[['x_1', 'y_1', 'z_1']].values
    return coords_dict

train_coords_dict = process_labels(train_labels)

# ---- TBM all-atom: load training CIF files from RCSB ----
TBM_CIF_DIR = Path('/kaggle/input/train_structures')

def _load_cif_atoms(pdb_id: str) -> dict | None:
    """Return {chain_id: {resid: {atom_name: np.array([x,y,z])}}} for one CIF."""
    cif_path = TBM_CIF_DIR / f'{pdb_id.upper()}.cif'
    if not cif_path.exists():
        return None
    try:
        st = gemmi.read_structure(str(cif_path))
        result = {}
        for model in st:
            for chain in model:
                by_res = {}
                for res in chain:
                    atoms = {}
                    for atom in res:
                        if atom.element != gemmi.Element('H'):
                            atoms[atom.name] = np.array(
                                [atom.pos.x, atom.pos.y, atom.pos.z], dtype=np.float32)
                    if atoms:
                        by_res[res.seqid.num] = atoms
                if by_res:
                    result[chain.name] = by_res
            break  # first model only
        return result or None
    except Exception as _e:
        return None

def _build_train_allatom(train_labels_df, train_seqs_df):
    """Build target_id -> per-residue all-atom dict (sorted by resid in sequence order).
    Falls back to None when CIF unavailable (TBM will use C1'-only fallback).
    """
    allatom = {}
    # Map target_id -> chain from labels
    chain_col = 'chain' if 'chain' in train_labels_df.columns else None
    prefixes = train_labels_df['ID'].str.rsplit('_', n=1).str[0]
    for tid, grp in train_labels_df.groupby(prefixes):
        pdb_id = tid[:4]
        cif_atoms = _load_cif_atoms(pdb_id)
        if cif_atoms is None:
            allatom[tid] = None
            continue
        # Determine chain(s) used for this target
        if chain_col:
            used_chains = grp[chain_col].unique().tolist()
        else:
            used_chains = list(cif_atoms.keys())[:1]
        # Collect residues in sequence order (sorted by resid)
        residues = []
        for ch in used_chains:
            if ch not in cif_atoms:
                continue
            for resid in sorted(cif_atoms[ch].keys()):
                residues.append(cif_atoms[ch][resid])  # dict {atom_name: xyz}
        allatom[tid] = residues if residues else None
    return allatom

print("Loading training all-atom data from CIF files...")
train_allatom_dict = _build_train_allatom(train_labels, train_seqs)
_n_cif = sum(1 for v in train_allatom_dict.values() if v is not None)
print(f"  All-atom loaded: {_n_cif}/{len(train_allatom_dict)} targets")

from Bio.Align import PairwiseAligner

aligner = PairwiseAligner()
aligner.mode = 'global'
aligner.match_score = 2
aligner.mismatch_score = -1.5

# Stronger gap penalties discourage "sliding" (critical: residue numbering must match)
aligner.open_gap_score   = -8
aligner.extend_gap_score = -0.4

# Also penalize terminal gaps (prevents end-gap semi-global behavior)
aligner.query_left_open_gap_score  = -8
aligner.query_left_extend_gap_score = -0.4
aligner.query_right_open_gap_score = -8
aligner.query_right_extend_gap_score = -0.4
aligner.target_left_open_gap_score = -8
aligner.target_left_extend_gap_score = -0.4
aligner.target_right_open_gap_score = -8
aligner.target_right_extend_gap_score = -0.4

def find_similar_sequences(query_seq, train_seqs_df, train_coords_dict, top_n=5):
    similar_seqs = []
    
    # Pre-filter: Iterate only valid targets
    # Note: aligner.score is much faster than generating full alignments
    for _, row in train_seqs_df.iterrows():
        target_id, train_seq = row['target_id'], row['sequence']
        if target_id not in train_coords_dict: continue
        
        # Length filter (keep your original logic)
        if abs(len(train_seq) - len(query_seq)) / max(len(train_seq), len(query_seq)) > 0.3: continue
        
        # FAST SCORE: Calculates score without traceback overhead
        raw_score = aligner.score(query_seq, train_seq)
        
        normalized_score = raw_score / (2 * min(len(query_seq), len(train_seq)))
        similar_seqs.append((target_id, train_seq, normalized_score, train_coords_dict[target_id]))
    
    similar_seqs.sort(key=lambda x: x[2], reverse=True)
    return similar_seqs[:top_n]

def adapt_template_to_query(query_seq, template_seq, template_coords):
    # Generate the alignment object
    # aligner.align returns an iterator; we take the first optimal alignment
    alignment = next(iter(aligner.align(query_seq, template_seq)))
    
    new_coords = np.full((len(query_seq), 3), np.nan)
    
    # VECTORIZED MAPPING:
    # alignment.aligned returns lists of (start, end) tuples for matched segments.
    # This avoids the slow python loop "for char_q, char_t in zip..."
    for (q_start, q_end), (t_start, t_end) in zip(*alignment.aligned):
        # Map the coordinate chunk directly
        t_chunk = template_coords[t_start:t_end]
        
        # Safety check to ensure shapes match (handles edge cases)
        if len(t_chunk) == (q_end - q_start):
            new_coords[q_start:q_end] = t_chunk

    # --- Interpolation Logic (Unchanged) ---
    for i in range(len(new_coords)):
        if np.isnan(new_coords[i, 0]):
            prev_v = next((j for j in range(i-1, -1, -1) if not np.isnan(new_coords[j, 0])), -1)
            next_v = next((j for j in range(i+1, len(new_coords)) if not np.isnan(new_coords[j, 0])), -1)
            if prev_v >= 0 and next_v >= 0:
                w = (i - prev_v) / (next_v - prev_v)
                new_coords[i] = (1-w)*new_coords[prev_v] + w*new_coords[next_v]
            elif prev_v >= 0: new_coords[i] = new_coords[prev_v] + [3, 0, 0]
            elif next_v >= 0: new_coords[i] = new_coords[next_v] + [3, 0, 0]
            else: new_coords[i] = [i*3, 0, 0]
            
    return np.nan_to_num(new_coords)

def adapt_template_to_query_allatom(query_seq, template_seq,
                                     template_allatom, template_c1):
    """Like adapt_template_to_query but copies all heavy atoms per residue.
    Returns list of dicts [{atom_name: xyz}] length=len(query_seq).
    Falls back to synthetic C1'-only entry where template has no all-atom data.
    """
    alignment = next(iter(aligner.align(query_seq, template_seq)))
    # Build query_pos -> template_pos mapping
    q2t = {}
    for (q_start, q_end), (t_start, t_end) in zip(*alignment.aligned):
        for dq, dt in zip(range(q_start, q_end), range(t_start, t_end)):
            q2t[dq] = dt

    result = []
    for qi in range(len(query_seq)):
        ti = q2t.get(qi)
        if ti is not None and template_allatom and ti < len(template_allatom):
            result.append(dict(template_allatom[ti]))  # copy atom dict
        else:
            # Gap or no CIF: fall back to C1' only
            c1 = template_c1[ti] if (ti is not None and ti < len(template_c1)) \
                 else np.array([qi * 3.0, 0.0, 0.0], dtype=np.float32)
            result.append({"C1'": c1.astype(np.float32)})
    return result


def write_tbm_pdb(allatom_residues, sequence, pdbfile):
    """Write a PDB from TBM all-atom residue list."""
    _fmt = '%6s%5d %4s %3s %1s%4d    %8.3f%8.3f%8.3f%6.2f%6.2f          %2s%2s'
    _elem = {'P':'P','O':'O','C':'C','N':'N','S':'S'}
    lines = ['REMARK TBM full-atom (from RCSB PDB template)']
    cnt = 1
    for i, atoms in enumerate(allatom_residues):
        nt = sequence[i] if i < len(sequence) else 'N'
        for aname, xyz in atoms.items():
            elem = _elem.get(aname[0], 'C')
            aname_fmt = (' ' + aname).ljust(4)[:4] if len(aname) < 4 else aname[:4]
            lines.append(_fmt % ('ATOM  ', cnt, aname_fmt, nt, 'A', i+1,
                float(xyz[0]), float(xyz[1]), float(xyz[2]), 0, 0, elem, ''))
            cnt += 1
    lines.append('END')
    with open(pdbfile, 'w') as _f:
        _f.write('\n'.join(lines))


def adaptive_rna_constraints(coordinates, target_id, confidence=1.0, passes=2):
    """
    Evaluation-driven constraints:
    - US-align is show-only rigid body => internal geometry errors are fatal
    - apply within each chain segment (no fake bond across chain breaks)
    """
    coords = coordinates.copy()
    segments = test_segs_map.get(target_id, [(0, len(coords))])

    # stronger corrections when confidence is low
    strength = 0.75 * (1.0 - min(confidence, 0.97))
    strength = max(strength, 0.02)

    for _ in range(passes):
        for (s, e) in segments:
            X = coords[s:e]
            L = e - s
            if L < 3:
                coords[s:e] = X
                continue

            # (1) bond i,i+1 to ~5.95Å (vectorized, symmetric)
            d = X[1:] - X[:-1]
            dist = np.linalg.norm(d, axis=1) + 1e-6
            target = 5.95
            scale = (target - dist) / dist
            adj = (d * scale[:, None]) * (0.22 * strength)
            X[:-1] -= adj
            X[1:]  += adj

            # (2) soft i,i+2 to ~10.2Å (vectorized, symmetric)
            d2 = X[2:] - X[:-2]
            dist2 = np.linalg.norm(d2, axis=1) + 1e-6
            target2 = 10.2
            scale2 = (target2 - dist2) / dist2
            adj2 = (d2 * scale2[:, None]) * (0.10 * strength)
            X[:-2] -= adj2
            X[2:]  += adj2

            # (3) Laplacian smoothing (removes kinks US-align cannot fix)
            lap = 0.5 * (X[:-2] + X[2:]) - X[1:-1]
            X[1:-1] += (0.06 * strength) * lap

            # (4) light self-avoidance (prevents steric collapse)
            if L >= 25:
                k = min(L, 160) if L > 220 else L
                if k < L:
                    idx = np.linspace(0, L - 1, k).astype(int)
                else:
                    idx = np.arange(L)

                P = X[idx]
                diff = P[:, None, :] - P[None, :, :]
                distm = np.linalg.norm(diff, axis=2) + 1e-6
                sep = np.abs(idx[:, None] - idx[None, :])

                mask = (sep > 2) & (distm < 3.2)
                if np.any(mask):
                    force = (3.2 - distm) / distm
                    vec = (diff * force[:, :, None] * mask[:, :, None]).sum(axis=1)
                    X[idx] += (0.015 * strength) * vec

            coords[s:e] = X

    return coords

def _rotmat(axis, ang):
    axis = np.asarray(axis, float)
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    x, y, z = axis
    c, s = np.cos(ang), np.sin(ang)
    C = 1.0 - c
    return np.array([
        [c + x*x*C,     x*y*C - z*s, x*z*C + y*s],
        [y*x*C + z*s,   c + y*y*C,   y*z*C - x*s],
        [z*x*C - y*s,   z*y*C + x*s, c + z*z*C]
    ], dtype=float)

def apply_hinge(coords, seg, rng, max_angle_deg=25):
    s, e = seg
    L = e - s
    if L < 30:
        return coords
    pivot = s + int(rng.integers(10, L - 10))
    axis = rng.normal(size=3)
    ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
    R = _rotmat(axis, ang)
    X = coords.copy()
    p0 = X[pivot].copy()
    X[pivot+1:e] = (X[pivot+1:e] - p0) @ R.T + p0
    return X

def jitter_chains(coords, segments, rng, max_angle_deg=12, max_trans=1.5):
    X = coords.copy()
    global_center = X.mean(axis=0, keepdims=True)
    for (s, e) in segments:
        axis = rng.normal(size=3)
        ang = np.deg2rad(float(rng.uniform(-max_angle_deg, max_angle_deg)))
        R = _rotmat(axis, ang)
        shift = rng.normal(size=3)
        shift = shift / (np.linalg.norm(shift) + 1e-12) * float(rng.uniform(0.0, max_trans))
        c = X[s:e].mean(axis=0, keepdims=True)
        X[s:e] = (X[s:e] - c) @ R.T + c + shift
    # recenter
    X -= X.mean(axis=0, keepdims=True) - global_center
    return X

def smooth_wiggle(coords, segments, rng, amp=0.8):
    X = coords.copy()
    for (s, e) in segments:
        L = e - s
        if L < 20:
            continue
        n_ctrl = 6
        ctrl_x = np.linspace(0, L - 1, n_ctrl)
        ctrl_disp = rng.normal(0, amp, size=(n_ctrl, 3))
        t = np.arange(L)
        disp = np.vstack([np.interp(t, ctrl_x, ctrl_disp[:, k]) for k in range(3)]).T
        X[s:e] += disp
    return X

def predict_rna_structures(row, train_seqs_df, train_coords_dict, n_predictions=5):
    tid = row['target_id']
    seq = row['sequence']

    # Data constraint: should already be canonical A/C/G/U
    assert set(seq).issubset(set("ACGU")), f"Non-ACGU in {tid}; do not remap here."

    segments = test_segs_map.get(tid, [(0, len(seq))])

    # Grab a larger candidate pool, then sample for diversity (best-of-5)
    cands = find_similar_sequences(query_seq=seq, train_seqs_df=train_seqs_df, train_coords_dict=train_coords_dict, top_n=30)
    assert all(len(c[3]) == len(c[1]) for c in cands), "Template coords/seq length mismatch"
    predictions = []
    used = set()

    for i in range(n_predictions):
        seed = (abs(hash(tid)) + i * 10007) % (2**32)
        rng = np.random.default_rng(seed)

        if not cands:
            # hard fallback (straight line per chain)
            coords = np.zeros((len(seq), 3), dtype=float)
            for (s, e) in segments:
                for j in range(s+1, e):
                    coords[j] = coords[j-1] + [5.95, 0, 0]
            predictions.append(coords)
            continue

        # Choose template:
        # i=0 => best template; others => sample among top-K with weights, avoid duplicates
        if i == 0:
            t_id, t_seq, sim, t_coords = cands[0]
        else:
            K = min(12, len(cands))
            sims = np.array([cands[k][2] for k in range(K)], float)
            w = np.exp((sims - sims.max()) / 0.08)
            # penalize already used templates
            for k in range(K):
                if cands[k][0] in used:
                    w[k] *= 0.10
            w = w / (w.sum() + 1e-12)
            k = int(rng.choice(np.arange(K), p=w))
            t_id, t_seq, sim, t_coords = cands[k]

        used.add(t_id)

        # Transfer coords with diagonal-guard mapping (no sliding)
        adapted = adapt_template_to_query(query_seq=seq, template_seq=t_seq, template_coords=t_coords)

        # Diversity transforms (then re-refine constraints)
        if i == 0:
            X = adapted
        elif i == 1:
            # mild noise
            X = adapted + rng.normal(0, max(0.01, (0.40 - sim) * 0.06), adapted.shape)
        elif i == 2:
            # hinge within the longest chain
            longest = max(segments, key=lambda se: se[1] - se[0])
            X = apply_hinge(adapted, longest, rng, max_angle_deg=22)
        elif i == 3:
            # inter-chain jitter (small, safe)
            X = jitter_chains(adapted, segments, rng, max_angle_deg=10, max_trans=1.0)
        else:
            # smooth low-frequency deformation
            X = smooth_wiggle(adapted, segments, rng, amp=0.7)

        refined = adaptive_rna_constraints(X, tid, confidence=sim, passes=2)
        predictions.append(refined)

    return predictions

all_predictions = []
start_time = time.time()
for idx, row in test_seqs.iterrows():
    if idx % 10 == 0: print(f"Processing {idx} | {time.time()-start_time:.1f}s")
    tid, seq = row['target_id'], row['sequence']
    preds = predict_rna_structures(row, train_seqs, train_coords_dict)
    for j in range(len(seq)):
        res = {'ID': f"{tid}_{j+1}", 'resname': seq[j], 'resid': j+1}
        for i in range(5):
            res[f'x_{i+1}'], res[f'y_{i+1}'], res[f'z_{i+1}'] = preds[i][j]
        all_predictions.append(res)

sub = pd.DataFrame(all_predictions)
cols = ['ID', 'resname', 'resid'] + [f'{c}_{i}' for i in range(1,6) for c in ['x','y','z']]

# Safety: competition clips coords; do it explicitly to avoid out-of-range explosions
coord_cols = [c for c in cols if c.startswith(('x_','y_','z_'))]
sub[coord_cols] = sub[coord_cols].clip(-999.999, 9999.999)

pred_tbm = sub


# print('Submission.csv shape:', rosetta_pred.shape)
pred_tbm.to_csv('/kaggle/working/pred_tbm.csv', index=False)
display(pred_tbm.head())

# ---- Write TBM full-atom PDB files ----
_TBM_STRUCT_DIR = Path('/kaggle/working/tbm_structures')
_TBM_STRUCT_DIR.mkdir(exist_ok=True)

for _, _trow in test_seqs.iterrows():
    _tid  = _trow['target_id']
    _qseq = _trow['sequence']
    _similar = find_similar_sequences(_qseq, train_seqs, train_coords_dict, top_n=5)
    if not _similar:
        continue
    _tdir = _TBM_STRUCT_DIR / _tid
    _tdir.mkdir(exist_ok=True)
    for _rank, (_tmpl_id, _tmpl_seq, _score, _tmpl_c1) in enumerate(_similar, 1):
        _tmpl_aa = train_allatom_dict.get(_tmpl_id)
        _aa_res = adapt_template_to_query_allatom(_qseq, _tmpl_seq, _tmpl_aa, _tmpl_c1)
        _pdb_path = str(_tdir / f'{_tid}_model_{_rank}.pdb')
        try:
            write_tbm_pdb(_aa_res, _qseq, _pdb_path)
        except Exception as _e:
            print(f'  TBM PDB failed {_tid} rank {_rank}: {_e}')
print(f"TBM structures written to {_TBM_STRUCT_DIR}")

In [ ]:
# ============================================================
# SECTION 4: RNApro Inference (seq_len < 1000, uses TBM templates)
# ============================================================

!cp -r /kaggle/input/datasets/theoviel/rnapro-src/RNAPro .
!cp /kaggle/input/datasets/theoviel/rnapro-src/rnapro-private-best-500m.ckpt .


%cd /kaggle/working/RNAPro


!pip install -e . --no-deps


DIST = "/kaggle/working/RNAPro/release_data/ccd_cache/"
!mkdir -p $DIST


# updated file paths
!cp /kaggle/input/datasets/jaejohn/rnapro-ccd-cache/ccd_cache/components.cif $DIST
!cp /kaggle/input/datasets/jaejohn/rnapro-ccd-cache/ccd_cache/components.cif.rdkit_mol.pkl $DIST


!pip install /kaggle/input/datasets/tobimichigan/biotite-1-2/biotite-1.2.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --no-deps


!python preprocess/convert_templates_to_pt_files.py --input_csv /kaggle/working/pred_tbm.csv --output_name templates.pt


IS_SCORING_RUN = os.environ.get('KAGGLE_IS_COMPETITION_RERUN')
print(IS_SCORING_RUN)


# %%python
df = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
df["sequence_len"] = df["sequence"].str.len()
# df = df[(df['sequence_len'] >= 200) & (df['sequence_len'] < 600)]
# df = df[df['sequence_len'] < 600]
df = df[df['sequence_len'] < 1000]
df.reset_index(drop=True, inplace=True)
print(df.shape)
if not IS_SCORING_RUN:
    # df = df.head(5)
    TEST_TARGETS = "8ZNQ,9ZCC"
    keep = {t.strip() for t in TEST_TARGETS.split(",") if t.strip()}
    df = df[df["target_id"].isin(keep)].reset_index(drop=True)
    print(df.shape)
    if df.empty:
        raise ValueError("TEST_TARGETS did not match any target_id values")

df.to_csv('/kaggle/working/sample_sequences.csv', index=False)

In [ ]:
%%writefile rnapro_inference_kaggle.sh

%export LAYERNORM_TYPE=torch # fast_layernorm, torch


# Inference parameters (RNAPro)
SEED=42
N_SAMPLE=1
N_STEP=200
N_CYCLE=10

# Paths
DUMP_DIR="../output"
# Set a valid checkpoint file path below
CHECKPOINT_PATH="../rnapro-private-best-500m.ckpt"

# Template/MSA settings
TEMPLATE_DATA="./release_data/kaggle/templates.pt"
# Note: template_idx supports 5 choices and maps to top-k:
# 0->top1, 1->top2, 2->top3, 3->top4, 4->top5
TEMPLATE_IDX=0
RNA_MSA_DIR="/kaggle/input/stanford-rna-3d-folding-2/MSA"

# SEQUENCES_CSV="/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"
SEQUENCES_CSV="/kaggle/working/sample_sequences.csv"

# RibonanzaNet2 path (keep as-is per request)
RIBONANZA_PATH="/kaggle/input/models/shujun717/ribonanzanet2/pytorch/alpha/1"

# Model selection: keep to an existing key to align defaults (N_step=200, N_cycle=10)
MODEL_NAME="rnapro_base"
mkdir -p "${DUMP_DIR}"

python3 runner/inference_kaggle.py \
    --model_name "${MODEL_NAME}" \
    --seeds ${SEED} \
    --dump_dir "${DUMP_DIR}" \
    --load_checkpoint_path "${CHECKPOINT_PATH}" \
    --use_msa true \
    --use_template "ca_precomputed" \
    --model.use_template "ca_precomputed" \
    --model.use_RibonanzaNet2 true \
    --model.template_embedder.n_blocks 2 \
    --model.ribonanza_net_path "${RIBONANZA_PATH}" \
    --template_data "${TEMPLATE_DATA}" \
    --template_idx ${TEMPLATE_IDX} \
    --rna_msa_dir "${RNA_MSA_DIR}" \
    --model.N_cycle ${N_CYCLE} \
    --sample_diffusion.N_sample ${N_SAMPLE} \
    --sample_diffusion.N_step ${N_STEP} \
    --load_strict true \
    --num_workers 0 \
    --triangle_attention "torch" \
    --triangle_multiplicative "torch" \
    --sequences_csv "${SEQUENCES_CSV}" \
    --max_len 1000

In [ ]:
%%writefile runner/inference_kaggle.py
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
# SPDX-License-Identifier: Apache-2.0

import os
import shutil
import logging
import traceback
import warnings
import argparse
from contextlib import nullcontext
from os.path import join as opjoin
from typing import Any, Mapping

import json
import torch
import pandas as pd
import numpy as np
import gc
from biotite.structure.io import pdbx

from configs.configs_base import configs as configs_base
from configs.configs_data import data_configs
from configs.configs_inference import inference_configs
from runner.dumper import DataDumper

from rnapro.config import parse_sys_args
from rnapro.config.config import ConfigManager, ArgumentNotSet
from rnapro.data.infer_data_pipeline import get_inference_dataloader
from rnapro.model.RNAPro import RNAPro
from rnapro.utils.distributed import DIST_WRAPPER
from rnapro.utils.seed import seed_everything
from rnapro.utils.torch_utils import to_device

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

logger = logging.getLogger(__name__)

logging.basicConfig(level=logging.INFO)
logging.getLogger("rnapro.data").setLevel(logging.INFO)
logging.getLogger("rnapro").setLevel(logging.INFO)


def parse_configs(configs: dict, arg_str: str = None, fill_required_with_null: bool = False):
    manager = ConfigManager(configs, fill_required_with_null=fill_required_with_null)
    parser = argparse.ArgumentParser()
    parser.add_argument("--max_len", type=int, default=1000, required=False)

    for key, (dtype, default_value, allow_none, required) in manager.config_infos.items():
        parser.add_argument("--" + key, type=str, default=ArgumentNotSet(), required=required)
    merged_configs = manager.merge_configs(vars(parser.parse_args(arg_str.split())) if arg_str else {})
    merged_configs.max_len = parser.parse_args(arg_str.split()).max_len
    return merged_configs

class dotdict(dict):
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__
    def __getattr__(self, name):
        try: return self[name]
        except KeyError: raise AttributeError(name)

class InferenceRunner(object):
    def __init__(self, configs: Any) -> None:
        self.configs = configs
        self.init_env()
        self.init_basics()
        self.init_model()
        self.load_checkpoint()
        self.init_dumper(need_atom_confidence=configs.need_atom_confidence, sorted_by_ranking_score=configs.sorted_by_ranking_score)

    def init_env(self) -> None:
        self.use_cuda = torch.cuda.device_count() > 0
        if self.use_cuda:
            self.device = torch.device("cuda:{}".format(DIST_WRAPPER.local_rank))
            torch.cuda.set_device(self.device)
        else:
            self.device = torch.device("cpu")

    def init_basics(self) -> None:
        self.dump_dir = self.configs.dump_dir
        self.error_dir = opjoin(self.dump_dir, "ERR")
        os.makedirs(self.dump_dir, exist_ok=True)
        os.makedirs(self.error_dir, exist_ok=True)

    def init_model(self) -> None:
        self.model = RNAPro(self.configs).to(self.device)

    def load_checkpoint(self) -> None:
        checkpoint_path = self.configs.load_checkpoint_path
        checkpoint = torch.load(checkpoint_path, self.device)
        sample_key = [k for k in checkpoint["model"].keys()][0]
        if sample_key.startswith("module."):
            checkpoint["model"] = {k[len("module."):]: v for k, v in checkpoint["model"].items()}
        self.model.load_state_dict(state_dict=checkpoint["model"], strict=True)
        self.model.eval()

    def init_dumper(self, need_atom_confidence: bool = False, sorted_by_ranking_score: bool = True):
        self.dumper = DataDumper(base_dir=self.dump_dir, need_atom_confidence=need_atom_confidence, sorted_by_ranking_score=sorted_by_ranking_score)

    @torch.no_grad()
    def predict(self, data: Mapping[str, Mapping[str, Any]]) -> dict[str, torch.Tensor]:
        eval_precision = {"fp32": torch.float32, "bf16": torch.bfloat16, "fp16": torch.float16}[self.configs.dtype]
        enable_amp = torch.autocast(device_type="cuda", dtype=eval_precision) if torch.cuda.is_available() else nullcontext()
        data = to_device(data, self.device)
        with enable_amp:
            prediction, _, _ = self.model(input_feature_dict=data["input_feature_dict"], label_full_dict=None, label_dict=None, mode="inference")
        return prediction

    def update_model_configs(self, new_configs: Any) -> None:
        self.model.configs = new_configs


def update_inference_configs(configs: Any, N_token: int):
    # Force enable AMP (skip_amp=False) to save memory
    configs.skip_amp.confidence_head = False
    configs.skip_amp.sample_diffusion = False
    return configs


def infer_predict(runner: InferenceRunner, configs: Any) -> None:
    try:
        dataloader = get_inference_dataloader(configs=configs)
    except Exception as e:
        logger.error(f"Dataloader error: {e}")
        traceback.print_exc()
        return

    # --- Patch for chunked inference ---
    if getattr(configs, "chunk_target_id", None) and getattr(configs, "chunk_span", None):
        chunk_id = configs.chunk_target_id
        original_id = chunk_id.split("_chk")[0]
        start, end = configs.chunk_span
        dataset = dataloader.dataset
        if hasattr(dataset, "template_features"):
            if original_id in dataset.template_features:
                orig_feats = dataset.template_features[original_id]
                # Slice template features
                chunk_feats = {}
                logger.info(f"Processing template features for {chunk_id} (span: {start}-{end})")
                
                for k, v in orig_feats.items():
                    if isinstance(v, (torch.Tensor, np.ndarray)):
                        try:
                            shape = v.shape
                            logger.info(f"  Feature '{k}': shape={shape}")
                            
                            # Case 1: (N, L, L, ...)
                            if len(shape) >= 3 and shape[1] == shape[2] and shape[1] >= end:
                                chunk_feats[k] = v[:, start:end, start:end]
                                logger.info(f"    -> Sliced dim 1,2: {chunk_feats[k].shape}")
                            # Case 2: (L, L, ...)
                            elif len(shape) >= 2 and shape[0] == shape[1] and shape[0] >= end:
                                chunk_feats[k] = v[start:end, start:end]
                                logger.info(f"    -> Sliced dim 0,1: {chunk_feats[k].shape}")
                            # Case 3: (N, L, ...)
                            elif len(shape) >= 2 and shape[1] >= end:
                                chunk_feats[k] = v[:, start:end]
                                logger.info(f"    -> Sliced dim 1: {chunk_feats[k].shape}")
                            # Case 4: (L, ...)
                            elif len(shape) >= 1 and shape[0] >= end:
                                chunk_feats[k] = v[start:end]
                                logger.info(f"    -> Sliced dim 0: {chunk_feats[k].shape}")
                            else:
                                chunk_feats[k] = v
                                logger.info(f"    -> Kept original")
                        except IndexError as e:
                            logger.warning(f"IndexError slicing template feature {k} for {chunk_id}: {e}")
                            chunk_feats[k] = v
                    else:
                        logger.info(f"  Feature '{k}': type={type(v)} (not sliced)")
                        chunk_feats[k] = v
                
                dataset.template_features[chunk_id] = chunk_feats
                logger.info(f"Mapped and sliced template for {chunk_id} from {original_id} ({start}:{end})")
            else:
                logger.warning(f"Original template {original_id} not found for chunk {chunk_id}")

    for seed in configs.seeds:
        seed_everything(seed=seed, deterministic=configs.deterministic)
        for batch in dataloader:
            try:
                data, atom_array, data_error_message = batch[0]
                sample_name = data["sample_name"]
                if len(data_error_message) > 0: continue

                new_configs = update_inference_configs(configs, data["N_token"].item())
                runner.update_model_configs(new_configs)
                prediction = runner.predict(data)
                runner.dumper.dump(
                    dataset_name="", pdb_id=sample_name, seed=seed,
                    pred_dict=prediction, atom_array=atom_array, entity_poly_type=data["entity_poly_type"]
                )
                
                # Explicitly cleanup memory
                del prediction
                del data
                torch.cuda.empty_cache()
                gc.collect()
                
            except Exception as e:
                logger.error(f"Inference error for {sample_name}: {e}")
                traceback.print_exc()
                if hasattr(torch.cuda, "empty_cache"): torch.cuda.empty_cache()
                gc.collect()


def make_dummy_solution(valid_df):
    solution = dotdict()
    for i, row in valid_df.iterrows():
        solution[row.target_id] = dotdict(target_id=row.target_id, sequence=row.sequence, coord=[])
    return solution

def solution_to_submit_df(solution):
    submit_df = []
    for k, s in solution.items():
        L = len(s.sequence)
        df = pd.DataFrame()
        df["ID"] = [f"{s.target_id}_{i + 1}" for i in range(L)]
        df["resname"] = list(s.sequence)
        df["resid"] = [i + 1 for i in range(L)]
        for j in range(len(s.coord)):
            df[f"x_{j+1}"] = s.coord[j][:, 0]
            df[f"y_{j+1}"] = s.coord[j][:, 1]
            df[f"z_{j+1}"] = s.coord[j][:, 2]
        submit_df.append(df)
    return pd.concat(submit_df)


def extract_c1_coordinates(cif_file_path):
    try:
        with open(cif_file_path, "r") as f:
            cif_data = pdbx.CIFFile.read(f)
        atom_array = pdbx.get_structure(cif_data, model=1)
        mask_c1 = np.char.strip(atom_array.atom_name.astype(str)) == "C1'"
        c1_atoms = atom_array[mask_c1]
        sort_indices = np.argsort(c1_atoms.res_id)
        return c1_atoms[sort_indices].coord
    except Exception as e:
        return None

# ==========================================
# 3D Assembly (Kabsch & Stitching) Functions
# ==========================================
def kabsch_umeyama(A, B):
    """ Kabsch algorithm to align chunk B onto chunk A """
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)
    AA = A - centroid_A
    BB = B - centroid_B
    H = BB.T @ AA
    U, S, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T
    t = centroid_A - R @ centroid_B
    return R, t

def stitch_chunks(chunks_coords, overlaps):
    """Stitch chunk coordinates using Kabsch alignment and linear interpolation over overlap regions."""
    final_coords = chunks_coords[0].copy()

    for i in range(1, len(chunks_coords)):
        curr_coords = chunks_coords[i]
        overlap = overlaps[i-1]

        A = final_coords[-overlap:]  # Tail portion of the previous chunk
        B = curr_coords[:overlap]    # Head portion of the current chunk

        # Compute rotation matrix and translation vector via Kabsch alignment
        R, t = kabsch_umeyama(A, B)

        # Apply the transformation to the entire chunk
        curr_coords_transformed = (curr_coords @ R.T) + t

        # Linearly interpolate over the overlap region (smoothing)
        weights = np.linspace(1, 0, overlap).reshape(-1, 1)
        blended_overlap = A * weights + curr_coords_transformed[:overlap] * (1 - weights)

        # Update final coordinates: overwrite overlap region and append the rest
        final_coords[-overlap:] = blended_overlap
        final_coords = np.vstack([final_coords, curr_coords_transformed[overlap:]])

    return final_coords

# ==========================================
# Modified runner processing
# ==========================================
def run_ptx(target_id, sequence, configs, template_idx, runner, chunk_span=None):
    temp_dir = f"./{configs.dump_dir}/input"
    os.makedirs(temp_dir, exist_ok=True)

    input_json = [{"sequences": [{"rnaSequence": {"sequence": sequence, "count": 1}}], "name": target_id}]
    input_json_path = os.path.join(temp_dir, f"{target_id}_input.json")
    with open(input_json_path, "w") as f:
        json.dump(input_json, f, indent=4)

    configs.input_json_path = input_json_path
    configs.template_idx = int(template_idx)
    
    # If target_id contains "_chk", pass it as chunk_target_id to trigger mapping in infer_predict
    if "_chk" in target_id:
        configs.chunk_target_id = target_id
        configs.chunk_span = chunk_span # Tuple (start, end)
    else:
        configs.chunk_target_id = None
        configs.chunk_span = None

    infer_predict(runner, configs)

    cif_file_path = f"{configs.dump_dir}/{target_id}/seed_42/predictions/{target_id}_sample_0.cif"
    coord = extract_c1_coordinates(cif_file_path)

    if coord is None:
        coord = np.zeros((len(sequence), 3), dtype=np.float32)
    elif coord.shape[0] < len(sequence):
        pad = np.zeros((len(sequence) - coord.shape[0], 3), dtype=np.float32)
        coord = np.concatenate([coord, pad], axis=0)

    return coord


def run() -> None:
    configs = {**configs_base, **{"data": data_configs}, **inference_configs}
    configs = parse_configs(configs=configs, arg_str=parse_sys_args(), fill_required_with_null=True)
    valid_df = pd.read_csv(configs.sequences_csv)
    runner = InferenceRunner(configs)
    solution = make_dummy_solution(valid_df)

    chunk_size = 500 # configs.max_len
    overlap_size = 100 # number of overlapping bases between chunks

    for idx, row in valid_df.iterrows():
        target_id = row.target_id
        sequence = row.sequence
        L = len(sequence)
        print(f"\n -> Processing {target_id} (Length: {L})")

        if L < 1000:
            # Short sequences: run inference directly without chunking
            for template_idx in range(2):
                coord = run_ptx(target_id, sequence, configs, template_idx, runner, chunk_span=(0, L))
                solution[target_id].coord.append(coord)
        else:
            # Long sequences: split into overlapping chunks for inference
            print(f"    Sequence exceeds max_len ({chunk_size}), triggering chunked prediction...")
            starts, ends = [], []
            start = 0
            while start < L:
                end = min(start + chunk_size, L)
                starts.append(start)
                ends.append(end)
                if end == L: break
                start = end - overlap_size

            for template_idx in range(1):
                chunk_coords = []
                for i in range(len(starts)):
                    c_seq = sequence[starts[i]:ends[i]]
                    c_id = f"{target_id}_chk{i}"
                    print(f"    -> Predicting chunk {i+1}/{len(starts)} ({starts[i]} to {ends[i]})")

                    # Run inference for each chunk
                    coord = run_ptx(c_id, c_seq, configs, template_idx, runner, chunk_span=(starts[i], ends[i]))
                    chunk_coords.append(coord)               

                # Compute overlap sizes and stitch chunks together
                overlaps = [ends[i-1] - starts[i] for i in range(1, len(starts))]
                final_coord = stitch_chunks(chunk_coords, overlaps)
                solution[target_id].coord.append(final_coord)

    print('\n\n -> Inference done! Saving to rnapro_submission.csv')
    submit_df = solution_to_submit_df(solution)
    submit_df = submit_df.fillna(0.0)
    submit_df.to_csv("/kaggle/working/rnapro_submission.csv", index=False)

if __name__ == "__main__":
    run()

In [ ]:
# run
!bash ./rnapro_inference_kaggle.sh

pred_rnapro = pd.read_csv("/kaggle/working/rnapro_submission.csv")
display(pred_rnapro.head())

%cd /kaggle/working

In [ ]:
# ============================================================
# SECTION 5: Boltz2 Inference (seq_len < 1000)
# ============================================================

subprocess.run(["cp", "-r", "/kaggle/input/datasets/lbugnon/boltz-src-minimal", "./"], check=False)
# Install boltz from src
subprocess.run(["pip", "install", "--no-index", "--no-build-isolation", "-e", "./boltz-src-minimal"], check=False)

subprocess.run(["pip", "install", "--no-index", "/kaggle/input/datasets/youhanlee/boltz-dependencies/mashumaro-3.14-py3-none-any.whl", "--no-deps"], check=False)
subprocess.run(["pip", "install", "--no-index", "/kaggle/input/datasets/youhanlee/boltz-dependencies/ihm-2.2-py3-none-any.whl", "--no-deps"], check=False)
subprocess.run(["pip", "install", "--no-index", "/kaggle/input/datasets/youhanlee/boltz-dependencies/modelcif-1.3-py3-none-any.whl", "--no-deps"], check=False)

# need to create tar so boltz does not try to download
os.makedirs("boltz_cache", exist_ok=True)
subprocess.run(["cp", "-r", "/kaggle/input/datasets/lbugnon/boltz2", "boltz_cache"], check=False)
!mv boltz_cache/boltz2/mols/mols boltz_cache/boltz2/mols_new
!rm -rf boltz_cache/boltz2/mols
!mv boltz_cache/boltz2/mols_new boltz_cache/boltz2/mols
shutil.rmtree("boltz_cache/boltz2/mols/mols", ignore_errors=True)
subprocess.run(["tar", "-cf", "boltz_cache/boltz2/mols.tar", "boltz_cache/boltz2/mols"], check=False)

MAX_LENGTH_BOLTZ = 1000

sequences = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
print(sequences.shape)

sequences["sequence_len"] = sequences["sequence"].str.len()

sequences = sequences[sequences['sequence_len'] < 1000]
sequences.reset_index(drop=True, inplace=True)

if not IS_SCORING_RUN:
  sequences = sequences.head(1)

print(sequences.shape)

print(max(sequences["sequence_len"].values))

In [ ]:
%%writefile /kaggle/working/patch_torch.py
import torch
import functools

# Monkey-patch torch.load to override safety settings
original_load = torch.load

@functools.wraps(original_load)
def patched_load(*args, **kwargs):
    # Force weights_only=False even when the library explicitly passes True
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)

# Apply the patch
torch.load = patched_load

print("✅ PyTorch Safety Check FORCE DISABLED (Overriding explicit arguments)")

In [ ]:
# Build YAML input files for Boltz2
def create_boltz_yaml(row, output_dir):

    data = {
        "id": str(row["target_id"]),
        "sequences": [
            {
                "rna": {
                    "id": str(row["stoichiometry"]).split(":")[0],
                    "sequence": str(row["sequence"])
                }
            }
        ]
    }

    # --- Ligand processing ---
    if pd.notna(row["ligand_SMILES"]):

        smiles_list = str(row["ligand_SMILES"]).split(";")
        ids_list = str(row["ligand_ids"]).split(";")

        ligands = []
        for s, i in zip(smiles_list, ids_list):
            ligands.append({
                "id": i.strip(),
                "smiles": s.strip()
            })

        if ligands:
            data["ligands"] = ligands

    # --- Write output ---
    file_path = Path(output_dir) / f"{row['target_id']}.yaml"

    with open(file_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(
            data,
            f,
            sort_keys=False,
            allow_unicode=True
        )

    return file_path

# Create YAML inputs for all test sequences
output_path = "/kaggle/working/inputs"
os.makedirs(output_path, exist_ok=True)

for _, row in sequences.iterrows():
    yaml_file = create_boltz_yaml(row, output_path)
print(f"Created")


for p in Path("/kaggle/working").glob("boltz_repeat_0*"):
    shutil.rmtree(p, ignore_errors=True)

os.makedirs("inputs", exist_ok=True)

max_tokens = 1000
max_repeats = 999
pred_repeats = 1


# Monkey-patch PyTorch Lightning Trainer to suppress logger errors in Kaggle
original_trainer_init = pl.Trainer.__init__

def patched_trainer_init(self, *args, **kwargs):
    # Disable logger to prevent errors in Kaggle environment
    kwargs["logger"] = False
    # Minimize log output beyond the progress bar
    kwargs["enable_checkpointing"] = False
    return original_trainer_init(self, *args, **kwargs)

# Apply the patch
pl.Trainer.__init__ = patched_trainer_init

for k in range(len(sequences)):

    t0 = time.time()
    name = sequences.iloc[k].target_id
    print(f"preparing {name} ({k+1} of {len(sequences)})")

    pred_seq = sequences.iloc[k].sequence

    # ignore super large seqs
    if len(sequences.iloc[k].sequence)>max_tokens:
        print(f"skipped {name}")
        continue

    # A for the chain to predict. First load all chains.
    try:
        def _detect_mol_type(seq: str) -> str:
            """Return 'protein', 'dna', or 'rna' based on sequence alphabet."""
            rna_dna = set("ACGUTNacgutn")
            non_nucleotide = set(seq.upper()) - set("ACGUTRYSWKMBDHVN")
            if non_nucleotide:
                return "protein"
            if "T" in seq.upper() and "U" not in seq.upper():
                return "dna"
            return "rna"

        chains, chain_ind = {}, 1
        cur_header = None
        for entry in sequences.iloc[k].all_sequences.split("\n"):
            if entry.startswith(">"):
                cur_header = entry
                repeats = min(len(entry.split("|")[1].split(",")) if "|" in entry else 1, max_repeats)
            else:
                mol_type = _detect_mol_type(entry)
                if entry == pred_seq:
                    # Target chain always gets index 0
                    chains[0] = entry, mol_type
                else:
                    # Keep non-target chains (protein or other RNA) for complex modelling
                    chains[chain_ind] = entry, mol_type
                    chain_ind += 1

        # Now keep the chain 0 plus all chains randomly picked to fit max_tokens
        filtered_chains, ntokens = {}, 0
        other_chains = [c for c in chains if c!=0]
        random.shuffle(other_chains)
        for c in [0] + other_chains:
            ntokens += len(chains[c][0])
            if ntokens >= max_tokens:
                break
            filtered_chains[c] = chains[c]
        chains = filtered_chains
    except:
        # use only target sequence
        chains = {0: (pred_seq, "rna")}

    # save yaml
    yaml_data = {
        "id": name,
        "sequences": []
    }

    # --- sequences ---
    for chain in sorted(chains.keys()):
        seq, seq_type = chains[chain]
        yaml_data["sequences"].append({
            seq_type: {
                "id": chr(ord("A") + chain),
                "sequence": seq
            }
        })

    # --- Ligands (only if present) ---
    row = sequences.iloc[k]

    if pd.notna(row["ligand_SMILES"]):
        smiles_list = str(row["ligand_SMILES"]).split(";")
        ids_list = str(row["ligand_ids"]).split(";")

        ligands = []
        for s, i in zip(smiles_list, ids_list):
            s = s.strip()
            i = i.strip()
            if s:
                ligands.append({
                    "id": i,
                    "smiles": s
                })

        if ligands:
            yaml_data["ligands"] = ligands

    # --- save ---
    with open(f"/kaggle/working/inputs/{name}.yaml", "w") as fout:
        yaml.safe_dump(yaml_data, fout, sort_keys=False)

    # print yaml content
    with open(f"/kaggle/working/inputs/{name}.yaml") as _f:
        print(_f.read())

    # --- setting ---
    _boltz_devices = int(os.environ.get("BOLTZ_DEVICES", "1"))
    boltz_args = {
        "diffusion_samples": 10,
        "diffusion_samples_affinity": 5,
        "devices": _boltz_devices,
        "num_workers": max(2, _boltz_devices * 2),
        "max_parallel_samples": max(1, _boltz_devices),
        "output_format": "pdb",
        "cache": "boltz_cache/boltz2/",
    }

    # --- inferring ---
    print(name, ":")
    print()
    for repeat in range(pred_repeats):
        cmd = (
            f"export PYTHONPATH=$PYTHONPATH:/kaggle/working && "
            f"python3 -c 'import patch_torch; from boltz.main import cli; cli()' predict /kaggle/working/inputs/{name}.yaml"
            f" --diffusion_samples {boltz_args['diffusion_samples']}"
            f" --diffusion_samples_affinity {boltz_args['diffusion_samples_affinity']}"
            f" --devices {boltz_args['devices']}"
            f" --num_workers {boltz_args['num_workers']}"
            f" --max_parallel_samples {boltz_args['max_parallel_samples']}"
            f" --output_format {boltz_args['output_format']}"
            f" --cache {boltz_args['cache']}"
            f" --out_dir boltz_repeat_{repeat}"
        )
        
        # Log prefix to distinguish runs
        print(f"Running with PyTorch 2.6 patch...")
        subprocess.run(cmd, shell=True, check=False)
        gc.collect()
        torch.cuda.empty_cache()


# ==========================================
# 1. Coordinate extraction
# ==========================================
def extract_coords(pdb_file, cif_file):
    """Extract C1' atom coordinates from mmCIF file."""
    try:
        structure = gemmi.read_structure(f"{pdb_file}")
        structure.make_mmcif_document().write_file(f"{cif_file}")
        mmcif_dict = MMCIF2Dict(cif_file)
        x_coords = mmcif_dict["_atom_site.Cartn_x"]
        y_coords = mmcif_dict["_atom_site.Cartn_y"]
        z_coords = mmcif_dict["_atom_site.Cartn_z"]
        atom_names = mmcif_dict["_atom_site.label_atom_id"]

        c1_coords = []
        for i, atom in enumerate(atom_names):
            if atom == "C1'":
                c1_coords.append([float(x_coords[i]), float(y_coords[i]), float(z_coords[i])])
        return np.array(c1_coords)
    except Exception as e:
        print(f"Error parsing {cif_file}: {e}")
        return None

# ==========================================
# 2. Model selection logic
# ==========================================

def get_top_n_predictions(diffusion_results_dir: Path, file_prefix: str, n: int = 3):
    """Return top-n predictions sorted by pLDDT and confidence score."""
    results = []
    target_subdir = diffusion_results_dir / file_prefix
    
    for i in range(10):
        json_path = target_subdir / f"confidence_{file_prefix}_model_{i}.json"
        if not json_path.exists():
            continue

        with open(json_path, 'r') as f:
            data = json.load(f)

        results.append({
            "idx": i,
            "plddt": data.get("complex_plddt", -float("inf")),
            "conf": data.get("confidence_score", -float("inf"))
        })

    # Sort by score (descending) and return top-n
    results.sort(key=lambda x: (x["plddt"], x["conf"]), reverse=True)
    return results[:n]


# ==========================================
# 3. Main processing
# ==========================================

def main_boltz():
    df_test = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
    submission = pd.read_csv('/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv')

    # Initialize all coordinate columns as float
    coord_cols = ['x_1', 'y_1', 'z_1',
                  'x_2', 'y_2', 'z_2',
                  'x_3', 'y_3', 'z_3',
                  'x_4', 'y_4', 'z_4',
                  'x_5', 'y_5', 'z_5',]
    submission[coord_cols] = submission[coord_cols].astype(float)

    for target_id, sequence in zip(df_test["target_id"], df_test["sequence"]):
        print(f'--- Processing {target_id} ---')
        diffusion_results_dir = Path(f"/kaggle/working/boltz_repeat_0/boltz_results_{target_id}/predictions")

        seq_len = len(sequence)
        top_models = get_top_n_predictions(diffusion_results_dir, target_id, n=5)

        # Mask rows corresponding to this target
        target_mask = submission['ID'].apply(lambda x: x.startswith(f"{target_id}_"))
        expected_count = target_mask.sum()

        # Store coordinates for each of the 5 predictions
        for rank, model_info in enumerate(top_models, start=1):
            m_idx = model_info["idx"]
            cif_file = diffusion_results_dir / target_id / f"{target_id}_model_{m_idx}.cif"
            pdb_file = diffusion_results_dir / target_id / f"{target_id}_model_{m_idx}.pdb"
            coords = extract_coords(pdb_file, cif_file)

            if coords is not None:
                # Pad with zeros if too few residues; truncate if too many
                if len(coords) < expected_count:
                    coords = np.vstack([coords, np.zeros((expected_count - len(coords), 3))])
                else:
                    coords = coords[:expected_count]

                # Assign to coordinate columns (x_N, y_N, z_N)
                cols = [f'x_{rank}', f'y_{rank}', f'z_{rank}']
                submission.loc[target_mask, cols] = coords
                print(f"  Rank {rank} (Model {m_idx}) loaded.")
            else:
                print(f"  Warning: Rank {rank} coordinates not found.")

    # Save output to CSV
    output_path = '/kaggle/working/boltz_submission.csv'
    submission.to_csv(output_path, index=False)
    print(f"\nFinal submission saved to {output_path}")

if __name__ == "__main__":
    main_boltz()


boltz_pred = pd.read_csv("/kaggle/working/boltz_submission.csv")
display(boltz_pred.head())

In [ ]:
# ============================================================
# SECTION 6: Ensemble & Final Submission
#
# Prediction slot assignment by sequence length:
#   seq_len < 250        : [Boltz2_1, Boltz2_2, RNApro_1, Protenix_1, DRFold2_1]
#   250 <= seq_len < 1000: [TBM_1,    Boltz2_1, RNApro_1, RNApro_2,   Boltz2_2 ]
#   seq_len >= 1000      : [TBM_1,    TBM_2,    TBM_3,    Protenix_1, Protenix_2]
# ============================================================

def overlay(merged, src_df, src_cols, dst_cols):
    """Write src_cols from src_df into dst_cols of merged (aligned by ID)."""
    merged = merged.set_index("ID")
    merged.loc[src_df["ID"], dst_cols] = src_df[src_cols].values
    return merged.reset_index(drop=False)


# --- Attach sequence_len to each prediction dataframe ---
# test_seqs already has sequence_len from Section 3
test_seqs["sequence_len"] = [len(i) for i in test_seqs["sequence"]]
seq_len_df = test_seqs[["target_id", "sequence_len"]]

def add_seq_len(df):
    df = df.copy()
    df["target_id"] = df["ID"].str.split("_").str[0]
    return df.merge(seq_len_df, on="target_id", how="left")

pred_tbm_      = add_seq_len(pred_tbm)
boltz_pred_    = add_seq_len(boltz_pred)
protenix_pred_ = add_seq_len(protenix_pred)
pred_rnapro_   = pred_rnapro.copy()  # seq_len < 1000 by construction (Section 4 filter)
drfold_pred_   = drfold_pred.copy()  # seq_len < 250  by construction (Section 2 filter)

SHORT = 250

# --- Step 1: Base layer (TBM for long seqs, Boltz2 for short seqs) ---
merged_pred = pd.concat([
    pred_tbm_  [pred_tbm_  ["sequence_len"] >= SHORT],
    boltz_pred_[boltz_pred_["sequence_len"] <  SHORT],
], axis=0)

# --- Step 2: RNApro → slot3 & slot4 (seq_len < 1000 only, by construction) ---
merged_pred = overlay(merged_pred, pred_rnapro_,
                      src_cols=["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"],
                      dst_cols=["x_3", "y_3", "z_3", "x_4", "y_4", "z_4"])

# --- Step 3: DRFold2 → slot5 (seq_len < 250 only, by construction) ---
merged_pred = overlay(merged_pred, drfold_pred_,
                      src_cols=["x_1", "y_1", "z_1"],
                      dst_cols=["x_5", "y_5", "z_5"])

# --- Step 4: Boltz2 → slot2 & slot5 (250 <= seq_len < 1000) ---
boltz_medium = boltz_pred_[
    (boltz_pred_["sequence_len"] >= SHORT) & (boltz_pred_["sequence_len"] < 1000)
]
merged_pred = overlay(merged_pred, boltz_medium,
                      src_cols=["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"],
                      dst_cols=["x_2", "y_2", "z_2", "x_5", "y_5", "z_5"])

# --- Step 5a: Protenix → slot4 (seq_len < 250) ---
protenix_short = protenix_pred_[protenix_pred_["sequence_len"] < SHORT]
merged_pred = overlay(merged_pred, protenix_short,
                      src_cols=["x_1", "y_1", "z_1"],
                      dst_cols=["x_4", "y_4", "z_4"])

# --- Step 5b: Protenix → slot4 & slot5 (seq_len >= 1000) ---
protenix_long = protenix_pred_[protenix_pred_["sequence_len"] >= 1000]
merged_pred = overlay(merged_pred, protenix_long,
                      src_cols=["x_1", "y_1", "z_1", "x_2", "y_2", "z_2"],
                      dst_cols=["x_4", "y_4", "z_4", "x_5", "y_5", "z_5"])

# --- Final: align to sample submission order and save ---
sub = pd.read_csv("/kaggle/input/stanford-rna-3d-folding-2/sample_submission.csv")["ID"]
sub = pd.merge(sub, merged_pred, how="left", on="ID").iloc[:, :18]
sub["resid"] = sub["resid"].astype(int)
sub.to_csv('/kaggle/working/submission.csv', index=False)
display(sub.head())

In [ ]:
# ============================================================
# SECTION 7: All-Atom Structure Output (CIF / PDB)
#
# Collect the full all-atom structures produced by each model
# and copy them to STRUCTURES_OUT (/kaggle/working/structures).
#
# Slot assignment mirrors the ensemble logic (Section 6):
#   seq_len <  250  : [Boltz2_1, Boltz2_2, RNApro_1, Protenix_1, DRFold2_1]
#   seq_len <  1000 : [TBM_1,   Boltz2_1, RNApro_1, RNApro_2,   Boltz2_2 ]
#   seq_len >= 1000 : [TBM_1,   TBM_2,   TBM_3,   Protenix_1, Protenix_2]
#
# Fallback chain: if a slot primary source is missing, try
# other available models, then repeat the best found file.
# ============================================================

import shutil
from math import inf as _INF

STRUCTURES_OUT = Path(os.environ.get("STRUCTURES_OUT", "/kaggle/working/structures"))
STRUCTURES_OUT.mkdir(parents=True, exist_ok=True)

# ---- Helpers to locate per-model output files ----

def _tbm_pdb(_tid: str, rank: int) -> Path | None:
    p = Path(f"/kaggle/working/tbm_structures/{_tid}") / f"{_tid}_model_{rank}.pdb"
    return p if p.exists() else None


def _protenix_cif(target_id: str, sample_idx: int) -> Path | None:
    p = Path("/kaggle/working/outputs") / target_id / "seed_42" / "predictions" / f"{target_id}_sample_{sample_idx}.cif"
    return p if p.exists() else None


def _rnapro_cif(target_id: str, sample_idx: int = 0) -> Path | None:
    p = Path("/kaggle/working/output") / target_id / "seed_42" / "predictions" / f"{target_id}_sample_{sample_idx}.cif"
    return p if p.exists() else None


def _boltz_pdb(target_id: str, rank: int) -> Path | None:
    """Return the rank-th best Boltz2 PDB (rank 1 = best confidence)."""
    pred_base = Path(f"/kaggle/working/boltz_repeat_0/boltz_results_{target_id}/predictions")
    target_sub = pred_base / target_id
    results = []
    for i in range(10):
        jpath = pred_base / f"confidence_{target_id}_model_{i}.json"
        if not jpath.exists():
            jpath = target_sub / f"confidence_{target_id}_model_{i}.json"
        if jpath.exists():
            try:
                with open(jpath) as _f:
                    d = json.load(_f)
                results.append({"idx": i,
                                 "score": (d.get("complex_plddt", -_INF),
                                           d.get("confidence_score", -_INF))})
            except Exception:
                pass
    results.sort(key=lambda x: x["score"], reverse=True)
    if rank - 1 < len(results):
        m_idx = results[rank - 1]["idx"]
        p = target_sub / f"{target_id}_model_{m_idx}.pdb"
        return p if p.exists() else None
    return None


def _drfold_pdb(target_id: str, rank: int) -> Path | None:
    p = Path(f"/kaggle/working/drfold_structures/{target_id}") / f"{target_id}_model_{rank}.pdb"
    return p if p.exists() else None


def _best_any(tid: str) -> Path | None:
    """Return any available structure for this target (last-resort fallback)."""
    for fn in [
        _boltz_pdb(tid, 1), _protenix_cif(tid, 0), _rnapro_cif(tid, 0),
        _tbm_pdb(tid, 1), _drfold_pdb(tid, 1),
        _protenix_cif(tid, 1), _rnapro_cif(tid, 1), _boltz_pdb(tid, 2),
        _tbm_pdb(tid, 2), _tbm_pdb(tid, 3),
    ]:
        if fn is not None:
            return fn
    return None


def _copy_structure(src: Path | None, dst: Path) -> bool:
    if src is not None and src.exists():
        shutil.copy2(src, dst)
        return True
    return False


# ---- Main export loop ----

_test_seqs_out = pd.read_csv(os.environ.get("TEST_CSV",
    "/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv"))

for _, _row in _test_seqs_out.iterrows():
    _tid = _row["target_id"]
    _slen = len(str(_row["sequence"]))
    _tdir = STRUCTURES_OUT / _tid
    _tdir.mkdir(exist_ok=True)

    if _slen < 250:
        # Short: Boltz2-dominant; Protenix slot 4; DRFold2 slot 5
        _sources = [
            _boltz_pdb(_tid, 1),                                   # slot 1
            _boltz_pdb(_tid, 2),                                   # slot 2
            _rnapro_cif(_tid, 0),                                  # slot 3
            _protenix_cif(_tid, 0),                                # slot 4
            _drfold_pdb(_tid, 1) or _boltz_pdb(_tid, 3),          # slot 5: DRFold2 preferred
        ]
    elif _slen < 1000:
        # Medium: TBM slot 1, Boltz2 slots 2+5, RNApro slots 3-4
        _sources = [
            _tbm_pdb(_tid, 1) or _boltz_pdb(_tid, 1) or _rnapro_cif(_tid, 0),  # slot 1
            _boltz_pdb(_tid, 1),                                   # slot 2
            _rnapro_cif(_tid, 0),                                  # slot 3
            _rnapro_cif(_tid, 1) or _rnapro_cif(_tid, 0),         # slot 4
            _boltz_pdb(_tid, 2),                                   # slot 5
        ]
    else:
        # Long (>=1000): TBM slots 1-3; Protenix slots 4-5 (samples 0 and 1)
        _sources = [
            _tbm_pdb(_tid, 1) or _protenix_cif(_tid, 0),          # slot 1
            _tbm_pdb(_tid, 2) or _protenix_cif(_tid, 1),          # slot 2
            _tbm_pdb(_tid, 3) or _protenix_cif(_tid, 0),          # slot 3
            _protenix_cif(_tid, 0),                                # slot 4
            _protenix_cif(_tid, 1) or _protenix_cif(_tid, 0),     # slot 5
        ]

    # Cross-model fallback: fill any remaining None slots with any available file
    _any = None
    for _i, _s in enumerate(_sources):
        if _s is None:
            if _any is None:
                _any = _best_any(_tid)
            _sources[_i] = _any  # may still be None if all models failed

    for _slot, _src in enumerate(_sources, 1):
        _ext = _src.suffix if _src is not None else ".cif"
        _dst = _tdir / f"{_tid}_model_{_slot}{_ext}"
        ok = _copy_structure(_src, _dst)
        if not ok:
            print(f"  [{_tid}] slot {_slot}: no all-atom file found (all models missing)")

print()
print(f"All-atom structures written to: {STRUCTURES_OUT}")
_all_files = list(STRUCTURES_OUT.rglob("*.cif")) + list(STRUCTURES_OUT.rglob("*.pdb"))
print(f"Total files: {len(_all_files)}")
